# LEAN-LLM-OPT — Few-shot Only v5

Based on v4, restore complete current-case CSV Observations as direct inputs to the
code-generation LLM on all six routes. Keep the v4 pandas table-text format:
`File: <path>` followed by `frame.to_string(index=False)`. Do not wrap the current
CSV data in JSON, generate an extraction plan, or inject preloaded data objects
into the generated program.

Keep ReAct for NRM/RA/TP/AP/FLP, the original two-stage Others workflow, few-shot
retrieval/rendering, full model outputs, solver/scoring, and cached 101 classifications.
Model text and the already-read Observation are both forwarded unchanged. Original
CSV paths remain available; generated programs still read the source files at runtime.
There is no symbolic-model filter, schema-only reader, or extra generation call.

Start a fresh kernel and use the original project environment. Automatic 101 execution
is enabled; optional experiments are disabled. Results use `outputs/few_shot_only_v5`.


## 1. Imports and configuration

All Python dependencies and shared settings are defined here. OpenAI calls require an API key; solving requires a Gurobi license.

Required packages: openai, python-dotenv, gurobipy, numpy, pandas, ipython, langchain-classic, langchain-community, langchain-core, langchain-openai, and faiss-cpu.


In [1]:
from __future__ import annotations
import ast
import fcntl
import gurobipy as gp
import hashlib
import json
import math
import numpy as np
import os
import openai
import httpx
import pandas as pd
import re
import tempfile
from IPython.display import display
from dotenv import load_dotenv
from functools import lru_cache
from langchain_classic.agents import AgentType, Tool, initialize_agent
from langchain_classic.chains import LLMChain, RetrievalQA
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_community.vectorstores import FAISS
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from pathlib import Path
from threading import RLock
from typing import Any
from urllib.parse import quote


In [ ]:
PROJECT_ROOT = Path(os.environ.get("LEAN_LLM_OPT_ROOT", Path.cwd())).expanduser().resolve()
BENCHMARK_PATH = PROJECT_ROOT / "Test_Dataset/Large-scale-or/Large-scale-or-101.csv"
OTHER_BENCHMARK_PATH = PROJECT_ROOT / "Test_Dataset/Small-scale/IndustryOR_fixedV2.csv"
RESULTS_DIR = PROJECT_ROOT / "outputs/few_shot_only_v5"
# Reuse only the baseline automatic predicted_label column for the 101-case ablation.
# For another benchmark, pass its matching automatic results.csv, or None to classify.
CLASSIFICATION_RESULTS_PATH = PROJECT_ROOT / "outputs/ap_matrix_fixed_cache_v2/automatic/results.csv"
load_dotenv(PROJECT_ROOT / ".env.local", override=False)
load_dotenv(PROJECT_ROOT / ".env", override=False)
NOTEBOOK_FILENAME = "Ablation_Study_Large-scale_Few-shot_Only_1018.ipynb"
candidate = Path(os.environ.get("LEAN_LLM_OPT_NOTEBOOK", NOTEBOOK_FILENAME)).expanduser()
NOTEBOOK_PATH = (candidate if candidate.is_absolute() else PROJECT_ROOT / candidate).resolve()
if not NOTEBOOK_PATH.is_file():
    raise FileNotFoundError(
        f"Notebook used for fingerprint does not exist: {NOTEBOOK_PATH}. "
        "Set LEAN_LLM_OPT_NOTEBOOK to the notebook being executed."
    )

user_api_key = os.environ.get("OPENAI_API_KEY")

VERSION = "few-shot-only-pandas-text-with-observation-codegen-101-v5"
MODEL_SNAPSHOT = "gpt-4.1-2025-04-14"
NRM_RETRY_ON_TRUNCATION = False  # False: propagate truncation immediately; True: retry once.
EMBEDDING_MODEL = "text-embedding-ada-002"
CLASS_LABELS = ["TP", "NRM", "RA", "FLP", "AP", "Mixture", "Others"]
WORKFLOW_ROUTES = ["TP", "NRM", "FLP", "AP", "RA", "Others"]
CSVQA_MODE = "pandas_only"  # One input-data path for every CSV route.
RAG_EXAMPLES_ALL_PATH = Path("Large_Scale_Or_Files/RAG_Examples_All.csv").expanduser()
RAG_EXAMPLES_ALL_PATH = (
    RAG_EXAMPLES_ALL_PATH if RAG_EXAMPLES_ALL_PATH.is_absolute()
    else PROJECT_ROOT / RAG_EXAMPLES_ALL_PATH
).resolve()
CACHE_SCHEMA_VERSION = "few-shot-only-pandas-text-with-observation-codegen-cache-v5"

# Describe the current executor truthfully; no enforced time/resource budget exists yet.
# When adding an isolated worker, wire its actual limits and immutable image digest
# into execution and these settings, and record that image in each result as well.
SOLVER_POLICY = "in_process_exec_require_optimal"
EXECUTION_LIMITS = {"wall_time_seconds": None, "solver_time_seconds": None}


MATCH_REL_TOL = 1e-4
MATCH_ABS_TOL = 1e-4

EXPERIMENT_METHOD = "few_shot_only_pandas_text_with_observation_codegen"


## 2. Shared model clients

All workflows use the configured model snapshot and output guard. Clients are created when requested.


In [3]:
HTTP_RETRY_EVENTS = []


def _record_http_request(request):
    index = int(request.headers.get("x-stainless-retry-count", "0"))
    if index:
        HTTP_RETRY_EVENTS.append({"endpoint": request.url.path, "retry_index": index})


@lru_cache(maxsize=1)
def _get_http_client():
    return httpx.Client(timeout=180, event_hooks={"request": [_record_http_request]})


class ModelOutputTruncated(RuntimeError):
    """The API explicitly reported an incomplete model response."""


class RejectTruncatedOutput(BaseCallbackHandler):
    raise_error = True

    def on_llm_end(self, response, **kwargs):
        for choices in response.generations:
            for generation in choices:
                metadata = getattr(getattr(generation, "message", None), "response_metadata", {}) or {}
                info = generation.generation_info or {}
                reason = str(info.get("finish_reason") or metadata.get("finish_reason") or "").lower()
                if reason in {"length", "max_tokens", "max_output_tokens"}:
                    error = ModelOutputTruncated(f"Model output is incomplete: finish_reason={reason}")
                    error.llm_usage = (response.llm_output or {}).get("token_usage", {})
                    raise error


def require_api_key():
    """Load credentials at use time, including after configuring a running kernel."""
    load_dotenv(PROJECT_ROOT / ".env.local", override=False)
    load_dotenv(PROJECT_ROOT / ".env", override=False)
    key = (os.environ.get("OPENAI_API_KEY") or user_api_key or "").strip()
    if not key:
        raise ValueError(
            "OPENAI_API_KEY is missing. Set it in the environment or in "
            f"{PROJECT_ROOT / '.env.local'} before running a test."
        )
    return key


def make_llm(*, timeout=180):
    return ChatOpenAI(
        model_name=MODEL_SNAPSHOT, openai_api_key=require_api_key(),
        temperature=0.0, top_p=1, n=1, max_retries=1, http_client=_get_http_client(), timeout=timeout,
        callbacks=[RejectTruncatedOutput()],
    )


def make_embeddings():
    return OpenAIEmbeddings(model=EMBEDDING_MODEL, openai_api_key=require_api_key(), max_retries=1, http_client=_get_http_client())


## 3. CSV loading and ReAct protocol

Read all supplied CSVs with pandas, preserving parsed rows, columns, source order,
identifiers and text values. No mode dispatch, filtering or evidence-model call is used.
The existing ReAct boundary and route-specific call-count requirements are retained.


In [4]:
def normalize_route(value: Any) -> str:
    text = str(value or "").strip().upper()
    aliases = {
        "NETWORK REVENUE MANAGEMENT": "NRM", "NETWORK REVENUE MANAGEMENT PROBLEM": "NRM",
        "RESOURCE ALLOCATION": "RA", "RESOURCE ALLOCATION PROBLEM": "RA",
        "TRANSPORTATION": "TP", "TRANSPORTATION PROBLEM": "TP",
        "ASSIGNMENT": "AP", "ASSIGNMENT PROBLEM": "AP",
        "FACILITY LOCATION": "FLP", "FACILITY LOCATION PROBLEM": "FLP",
    }
    text = aliases.get(text, text)
    if text in {"MIXTURE", "OTHER", "OTHERS"} or text.startswith("OTHERS"):
        return "Others"
    if text == "UFLP":
        return "FLP"
    if text in {"NRM", "RA", "TP", "AP", "FLP"}:
        return text
    raise ValueError(f"Unsupported execution route: {value!r}")



def invoke_react_with_required_csvqa(agent, query, route):
    """Require CSVQA while preserving the existing ReAct agent boundary."""
    def call_count(result):
        return sum(
            isinstance(step, tuple)
            and len(step) >= 2
            and getattr(step[0], "tool", None) == "CSVQA"
            for step in result.get("intermediate_steps", [])
        )

    result = agent.invoke(query)
    if call_count(result) == 0:
        result = agent.invoke(
            "Protocol correction: call CSVQA before the Final Answer. Begin with Action: CSVQA.\n\n"
            f"User Description: {query}"
        )
    calls = call_count(result)
    exactly_once = normalize_route(route) in {"NRM", "RA"}
    if exactly_once and calls != 1:
        raise RuntimeError(f"{route} ReAct must call CSVQA exactly once; observed {calls}")
    if not exactly_once and calls < 1:
        raise RuntimeError(f"{route} ReAct must call CSVQA at least once; observed {calls}")
    return result



def read_csv_compat(path, **kwargs):
    """Read a project-relative or absolute CSV with the requested pandas options."""
    path = Path(path).expanduser()
    path = path if path.is_absolute() else PROJECT_ROOT / path
    last_error = None
    for encoding in ("utf-8-sig", "utf-8", "gbk", "latin-1"):
        try:
            return pd.read_csv(path, encoding=encoding, **kwargs)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise UnicodeError(f"Unable to decode CSV: {path}") from last_error



def _read_csv(path):
    """Preserve every parsed source row, column and text value, including index-like IDs."""
    frame = read_csv_compat(path, dtype=str, keep_default_na=False)
    frame.columns = [str(column).lstrip("\ufeff") for column in frame.columns]
    return frame



def _load_tables(dataset_address):
    """Resolve one-or-more newline-separated CSV paths in source order."""
    tables = []
    for raw in str(dataset_address or "").splitlines():
        raw = raw.strip()
        if not raw:
            continue
        path = Path(raw).expanduser()
        path = path if path.is_absolute() else (PROJECT_ROOT / path).resolve()
        if not path.is_file():
            raise FileNotFoundError(f"Cannot find input CSV file: {path}")
        tables.append({"file_index": len(tables), "path": path, "frame": _read_csv(path)})
    if not tables:
        raise ValueError("At least one input CSV path is required")
    return tables


## 4. Complete CSV observations as pandas table text

Current-case CSV observations use `File: <path>` followed by `frame.to_string(index=False)`.
All files, rows and columns remain in source order; the DataFrame display index is omitted,
not a source CSV column. The reader uses `dtype=str, keep_default_na=False` and does not
select rows, calculate aggregates, or wrap the data in JSON. It does not use `str(frame)`
or `head()`. Subsets and eligibility remain the modeling agent's task.

Historical few-shot rendering, retrieval metadata, trace logs and result/cache formats are
unchanged. They may still use JSON; they are not the current case's CSV Tool Observation.


In [5]:
def load_csv_data_result(dataset_address):
    """Read complete CSVs with pandas and return ordinary table text, not JSON."""
    tables = _load_tables(dataset_address)
    observations = []
    for table in tables:
        frame, path = table["frame"], table["path"]
        observations.append(f"File: {path}\n{frame.to_string(index=False)}")
    observation = "\n\n".join(observations)
    return {
        "observation": observation,
        "trace": {
            "method": EXPERIMENT_METHOD, "mode": CSVQA_MODE,
            "status": "PANDAS_FULL_DATA", "planner_attempt_count": 0,
            "extraction_llm_calls": 0,
            "table_count": len(tables),
            "row_counts": [len(table["frame"]) for table in tables],
            "payload_hash": hashlib.sha256(observation.encode("utf-8")).hexdigest(),
        },
    }


## 5. Pandas-only CSVQA tool

`Action: CSVQA` is a compatibility name. The tool ignores the natural-language
retrieval request and returns all data from the case's bound CSV paths. It never
calls an LLM, builds an index, filters rows, or silently limits observation size.
Repeated calls within one formulation reuse the same complete observation.


In [6]:
CSV_DATA_TOOL_DESCRIPTION = (
    "Read all CSV files supplied with the current problem using pandas and return "
    "every row and column as pandas table text in original source order. This tool does not answer "
    "questions, select subsets, calculate statistics, or summarize data. "
    "Apply the original query's selection and eligibility rules in the model."
)


def build_csvqa_components(dataset_address):
    """Build a pure pandas reader; the modeling LLM is created by its caller."""
    state = {}

    def read_all_csvs(_tool_query):
        # Paths are bound to the case, never taken from the agent's action input.
        if not state:
            state.update(load_csv_data_result(dataset_address))
        trace = state["trace"]
        trace["tool_call_count"] = trace.get("tool_call_count", 0) + 1
        return state["observation"]

    return Tool(name="CSVQA", func=read_all_csvs,
                description=CSV_DATA_TOOL_DESCRIPTION), state


## 6. Problem classification and baseline-label reuse

The existing classifier is retained for benchmarks run with `classification_csv=None`.
The 101-case automatic experiment instead joins baseline predictions by `problem_id`.
Only prediction and identity/provenance columns are read from the classification CSV;
no gold label, model, code, objective, or solver-output column is read from that file.


In [7]:
@lru_cache(maxsize=1)
def _get_classifier_retriever():
    documents = CSVLoader(
        file_path=str(RAG_EXAMPLES_ALL_PATH),
        encoding="utf-8-sig",
        content_columns=["prompt", "Type"],
    ).load()
    if len(documents) < 5:
        raise ValueError("Classification reference requires at least five labeled examples")
    return FAISS.from_documents(documents, make_embeddings()).as_retriever(
        search_kwargs={"k": 5}
    )


def fileqa_retrieve(query):
    """Retrieve five raw classification examples without an additional LLM call.
    
    Args:
        query: Natural-language optimization problem or retrieval query.
    
    Returns:
        A formatted string containing the ranked retrieved examples.
    """
    retrieved = _get_classifier_retriever().invoke(str(query))
    if len(retrieved) != 5:
        raise RuntimeError(f"FileQA expected five examples, retrieved {len(retrieved)}")

    return "\n\n".join(
        f"[Retrieved example {rank}]\n{document.page_content}"
        for rank, document in enumerate(retrieved, start=1)
    )


few_shot_example = """
Example 1 — Network Revenue Management

Question:
What is the problem type of the following optimization problem? Text: There
are three products with corresponding revenues and independent demand
streams. The company decides how many requests to fulfil over a finite sales
horizon to maximize total revenue from fixed initial inventories.
Replenishment is not allowed, and fulfilment decisions are non-negative
integers.

Thought: I need to identify the mathematical structure. I should call FileQA
once to retrieve structurally similar labeled reference problems.

Action: FileQA

Action Input: finite-horizon revenue maximization with independent demand streams, fixed initial inventories, no replenishment, and integer demand-fulfilment decisions

Observation: The retrieved RAG_Examples_All examples contain finite-inventory
demand-fulfilment problems labeled Network Revenue Management.

Thought: Fixed inventories are allocated to demand fulfilment over a sales
horizon to maximize revenue. This is the defining NRM structure.

Final Answer: NRM


Example 2 — Mixture rather than Resource Allocation

Question:
What is the problem type of the following optimization problem? Text: A
manufacturer determines integer production quantities and integer batch
counts on several workstations. Each batch consumes machine capacity.
Production creates a byproduct that must either be sold or disposed of, and
the corresponding quantities must satisfy balance constraints.

Thought: This problem contains resource capacities, but I must check whether
it is a clean single-stage resource-allocation model or contains additional
interacting mechanisms. I should call FileQA once.

Action: FileQA

Action Input: integer production and batch-count decisions across multiple workstations with capacity constraints and coupled byproduct sale-or-disposal balance

Observation: The retrieved examples include resource-allocation and
non-standard production-planning models. The latter contain batch decisions,
multiple production components, and coupled balance constraints.

Thought: Ordinary capacity constraints alone could indicate RA. However,
integer batch counts, multiple workstations, and byproduct balance materially
change the formulation. Therefore, the problem is Mixture rather than RA.

Final Answer: Mixture


Example 3 — Others rather than Mixture

Question:
What is the problem type of the following optimization problem? Text: A
company selects a subset of independent projects. Each project has a profit
and consumes part of one available budget. Each project is represented by a
binary variable, and there are no precedence, dependency, assignment, batch,
flow, or multi-stage constraints.

Thought: I need to determine whether this is a hybrid model or a coherent
standard optimization family outside the five canonical classes. I should
call FileQA once.

Action: FileQA

Action Input: independent binary project selection maximizing profit under one budget constraint with no logical dependencies or additional coupling

Observation: The retrieved examples contain standard binary knapsack and
project-selection problems categorized as outside the five canonical
classes.

Thought: This is a clean knapsack-style model without interacting structures
or non-standard coupling. It should be Others, not Mixture.

Final Answer: Others
"""


prefix = f"""
You are the classification agent in LEAN-LLM-OPT. Classify the current
optimization problem into exactly one of the following seven semantic
classes:

- NRM: Network Revenue Management
- RA: Resource Allocation
- TP: Transportation Problem
- FLP: Facility Location Problem
- AP: Assignment Problem
- Mixture
- Others

You must follow the ReAct workflow and call FileQA exactly once.

FileQA retrieves structurally similar labeled optimization problems from
RAG_Examples_All. For the Action Input, provide a concise but complete structural
description that preserves the objective, decision variables, major
constraints, and important coupling mechanisms.

Retrieved examples are supporting evidence, not automatic votes. The current
problem description and its mathematical structure are authoritative.

Classify according to:

- the objective;
- the meanings and domains of the decision variables;
- the principal constraint families;
- coupling between decisions;
- whether the formulation contains multiple stages or additional modeling
  mechanisms.

Do not classify according to company names, application narratives,
filenames, CSV files, or isolated keywords.

Class definitions:

1. Network Revenue Management (NRM)

A problem that allocates fixed inventories or capacities to known or
stochastic demand streams in order to maximize revenue. Typical defining
features include demand fulfilment, finite inventory, a sales horizon, and
no replenishment.

2. Resource Allocation (RA)

A clean, primarily single-stage allocation of limited resources among
activities or products, usually with ordinary capacity constraints and an
additive objective.

Capacity constraints or integer production quantities alone do not make a
problem Mixture.

Do not choose RA when additional mechanisms materially change the
formulation, such as:

- batch-size or lot-count decisions;
- multiple interacting production stages, workstations, or devices;
- byproduct sale, disposal, or material-balance decisions;
- blending or proportion constraints interacting with production;
- logical dependencies or conditional decisions;
- selection followed by assignment;
- equilibrium-style balance equations;
- vehicle-count and shipment-quantity coupling.

3. Transportation Problem (TP)

A clean source-to-destination flow problem with source supplies, destination
demands, route shipment quantities, and route transportation costs.

Do not choose TP when the transportation structure is materially coupled
with mechanisms such as integer truck dispatch, vehicle-capacity decisions,
fixed-charge decisions, multi-stage transshipment, production decisions, or
other non-standard constraint families. Such a problem may be Mixture.

4. Facility Location Problem (FLP)

A problem containing facility-opening or location decisions with fixed
opening costs, together with assignment or shipment decisions that link
clients or demand points to opened facilities.

5. Assignment Problem (AP)

A clean one-to-one or matching-style assignment of agents, workers,
machines, products, or other entities to tasks.

If the problem first selects a subset of entities and then assigns them, or
combines assignment with multiple stages, logical dependencies, production,
flow, or other interacting constraint families, choose Mixture instead.

6. Mixture

A hybrid or non-standard optimization model in which multiple structures or
modeling mechanisms interact materially.

Choose Mixture when either:

(a) a canonical TP, NRM, RA, FLP, or AP structure is combined with an
important additional mechanism; or

(b) a non-core optimization model contains non-standard interacting
constraint families that prevent it from being a clean textbook instance.

Strong Mixture signals include:

- batch-size, lot-size, or integer-trip decisions coupled with quantities;
- logical implications, precedence, dependencies, or conditional decisions;
- subset selection coupled with assignment;
- blending or proportion requirements coupled with production or capacity;
- byproduct production, sale, disposal, or balance;
- multiple interacting workstations, devices, or production stages;
- equilibrium or cross-variable balance equations;
- truck-count or vehicle-capacity decisions coupled with transportation flow;
- several distinct decision families whose interactions materially affect
  the formulation.

A problem does not need to belong to two named canonical classes to be
Mixture. One recognizable base structure plus a material non-standard
coupling mechanism is sufficient.

7. Others

A clean and coherent optimization family outside NRM, RA, TP, FLP, and AP,
without the hybrid or non-standard interacting structure required for
Mixture.

Typical examples include:

- a standard diet problem;
- a pure knapsack or independent project-selection problem;
- a standard traveling salesperson problem;
- a standard workforce or shift-covering problem;
- a standard parallel-machine scheduling problem;
- another well-known outside family with one dominant mathematical
  structure.

A textbook blending model may be Others only when it is a standalone model
with ordinary ingredient-balance and quality constraints. If blending or
proportion decisions interact with production stages, capacities, equipment,
batch decisions, or other constraint families, choose Mixture.

Decision procedure:

1. Identify the objective, decision variables, and main constraints.

2. Determine whether the problem is a clean instance of NRM, RA, TP, FLP,
   or AP.

3. Even if a canonical base structure is present, check for a material
   additional mechanism such as batching, logical dependencies,
   selection-assignment coupling, byproducts, blending-production coupling,
   multiple stages, equilibrium equations, or vehicle-count coupling.

4. If such an additional mechanism materially changes the formulation,
   choose Mixture.

5. If the problem is a clean, well-known outside family with one dominant
   structure and no material hybrid coupling, choose Others.

6. When TP, NRM, RA, FLP, or AP cannot be confidently confirmed:
   - choose Mixture if multiple structures or non-standard constraints
     interact materially;
   - otherwise choose Others.

When deciding between Mixture and Others, choose Mixture if two different decision families or constraint mechanisms interact materially. Choose Others only when one standard textbook archetype explains all important variables and constraints.

7. Retrieved labels are evidence, not rules. Do not use majority voting.
   When retrieval conflicts with the current mathematical formulation,
   prioritize the current formulation.

Important boundary rules:

- Multiple ordinary resource-capacity constraints alone are not sufficient
  for Mixture; a clean allocation model remains RA.
- Integer variables alone are not sufficient for Mixture.
- A canonical structure plus a material additional decision mechanism is
  sufficient for Mixture.
- Mixture and Others are separate semantic labels even if they currently
  share the same downstream workflow route.

Return exactly one of the following label tokens in the Final Answer:

NRM, RA, TP, FLP, AP, Mixture, Others

Do not include explanations, punctuation, or multiple candidate labels in
the Final Answer.

Examples of the required ReAct process:

{few_shot_example}
"""


suffix = """
Begin!

Question: {input}
{agent_scratchpad}
"""


@lru_cache(maxsize=1)
def _get_classification_agent():
    llm = make_llm()
    tool = Tool(
        name="FileQA",
        func=fileqa_retrieve,
        description=(
            "Return five structurally similar RAG examples and their labels. "
            "Use the complete optimization problem text as input."
        ),
    )
    return initialize_agent(
        tools=[tool], llm=llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
        agent_kwargs={"prefix": prefix, "suffix": suffix}, verbose=True,
        handle_parsing_errors=True, max_iterations=4, return_intermediate_steps=True,
    )


def normalize_problem_class(value):
    """Normalize a raw label to one of the seven benchmark classes.
    
    Args:
        value: Input value to normalize, parse, or inspect.
    
    Returns:
        The canonical class label.
    """
    candidate = str(value or "").strip()
    final = re.findall(r"(?:Final Answer|Answer|Label|Class)\s*:\s*([^\n]+)", candidate, re.I)
    if final:
        candidate = final[-1].strip()
    upper = re.sub(r"\s+", " ", candidate.strip("`*_ .,:;").upper())
    aliases = {
        "TP": "TP", "TRANSPORTATION": "TP", "TRANSPORTATION PROBLEM": "TP",
        "NRM": "NRM", "NETWORK REVENUE MANAGEMENT": "NRM",
        "NETWORK REVENUE MANAGEMENT PROBLEM": "NRM",
        "RA": "RA", "RESOURCE ALLOCATION": "RA", "RESOURCE ALLOCATION PROBLEM": "RA",
        "FLP": "FLP", "UFLP": "FLP", "FACILITY LOCATION": "FLP",
        "FACILITY LOCATION PROBLEM": "FLP",
        "UNCAPACITATED FACILITY LOCATION PROBLEM": "FLP",
        "UNCAPACITED FACILITY LOCATION PROBLEM": "FLP",
        "AP": "AP", "ASSIGNMENT": "AP", "ASSIGNMENT PROBLEM": "AP",
        "MIXTURE": "Mixture", "OTHERS": "Others",    }
    if upper in aliases:
        return aliases[upper]
    if upper.startswith("OTHERS") or upper.startswith("SALES-BASED LINEAR PROGRAMMING"):
        return "Others"
    matches = [
        label for label, pattern in [
            ("Mixture", r"\bMIXTURE\b"),
            ("NRM", r"\bNRM\b|NETWORK REVENUE MANAGEMENT"),
            ("RA", r"\bRA\b|RESOURCE ALLOCATION"),
            ("TP", r"\bTP\b|TRANSPORTATION"),
            ("FLP", r"\bFLP\b|\bUFLP\b|FACILITY LOCATION"),
            ("AP", r"\bAP\b|ASSIGNMENT"),
            ("Others", r"\bOTHERS"),
        ]
        if re.search(pattern, upper)
    ]
    if len(matches) == 1:
        return matches[0]
    raise ValueError(f"Cannot unambiguously normalize class: {value!r}")


def class_to_workflow_route(label):
    """Map a semantic class to an executable workflow route.
    
    Args:
        label: Semantic problem-class label.
    
    Returns:
        The route name; Mixture and Others both map to Others.
    """
    normalized = normalize_problem_class(label)
    return "Others" if normalized in {"Mixture", "Others"} else normalized


def invoke_classifier(query):
    """Run the ReAct classification agent and validate its tool-use contract.
    
    Args:
        query: Natural-language optimization problem or retrieval query.
    
    Returns:
        A dictionary containing the raw output, normalized label, FileQA call count, and trace.
    
    Raises:
        RuntimeError: If the classifier violates the one-FileQA-call contract or emits an invalid label.
    """
    response = _get_classification_agent().invoke(str(query))
    steps = response.get("intermediate_steps", []) if isinstance(response, dict) else []
    fileqa_calls = sum(
        getattr(step[0], "tool", None) == "FileQA"
        for step in steps if isinstance(step, tuple) and step
    )
    if fileqa_calls != 1:
        raise RuntimeError(f"Classifier must call FileQA exactly once; observed {fileqa_calls}")
    output = str(response.get("output", "") if isinstance(response, dict) else response).strip()
    return {
        "output": output,
        "normalized_label": normalize_problem_class(output),
        "fileqa_calls": fileqa_calls,
        "intermediate_steps": steps,
    }



def attach_cached_classifications(frame, classification_csv):
    """Join automatic baseline predictions by ID; never read gold/model/solver columns."""
    frame = prepare_cases(frame)
    path = Path(classification_csv).expanduser()
    path = (path if path.is_absolute() else PROJECT_ROOT / path).resolve()
    required = {"problem_id", "predicted_label"}
    columns = required | {"query", "experiment_mode", "forced_route"}
    cached = read_csv_compat(path, dtype=str, keep_default_na=False,
                             usecols=lambda name: name in columns)
    missing_columns = required.difference(cached.columns)
    if missing_columns:
        raise ValueError(f"Classification CSV missing columns: {sorted(missing_columns)}")
    if cached["problem_id"].str.strip().eq("").any() or cached["problem_id"].duplicated().any():
        raise ValueError("Classification CSV requires unique, nonempty problem_id values")
    if "experiment_mode" in cached and not cached["experiment_mode"].eq("automatic").all():
        raise ValueError("Use baseline automatic results.csv for classification reuse")
    if "forced_route" in cached and cached["forced_route"].ne("").any():
        raise ValueError("Classification CSV contains forced-route results")
    lookup = cached.set_index("problem_id")
    missing = frame.loc[~frame["problem_id"].isin(lookup.index), "problem_id"].tolist()
    if missing:
        raise ValueError(f"Missing baseline predictions for: {missing}")
    matched = lookup.loc[frame["problem_id"]]
    if "query" in matched and matched["query"].tolist() != frame["Query"].astype(str).tolist():
        raise ValueError("Classification CSV questions do not match the selected problem IDs")
    result = frame.copy()
    result["cached_predicted_label"] = matched["predicted_label"].map(normalize_problem_class).tolist()
    return result


def classification_for_case(case):
    if "cached_predicted_label" not in case:
        return {**invoke_classifier(str(case["Query"])),
                "classification_source": "online_classifier"}
    return {
        "normalized_label": normalize_problem_class(case["cached_predicted_label"]),
        "classification_source": "baseline_results_csv",
    }


## 7. Route retrieval and NRM formulation

Retain the same retrieved few-shot examples and ReAct modeler. NRM receives complete
pandas-read tables, with no extraction plan or pre-applied filter; its symbolic Data
Mapping must express the original query's selection explicitly.


In [8]:
QUERY_SEMANTICS_GUIDANCE = '\nThe original query is authoritative when a reference example or intermediate plan conflicts with it.\nEligibility or authorization permits a choice; it does not force selection. When an integer\nquantity may be zero or within [L,U], use a binary selected flag with L*selected <= quantity\n<= U*selected, and enforce positive quantity iff selected (quantity >= selected when L is zero).\nAn activation flag means iff its stated condition holds: an item flag iff positive quantity;\na group flag iff at least one member is selected (member_flag <= group_flag for every member,\ngroup_flag <= sum(member_flags)); a pair flag iff both members are selected (the two upper\nbounds and the lower bound). Enforce both directions regardless of fee/bonus signs.\nKeep unconditional aggregate bounds unconditional. Do not add restrictions, lazy constraints,\nfixed-zero decisions, or mandatory selections without support in the current query.\n'


AP_ELIGIBILITY_GUIDANCE = """\nFor AP, eligibility is a decision-pair mask separate from cost availability. A nonblank cost\non an ineligible pair is valid source data; keep the cost row and eligibility fields, and do not\nreject the dataset because that pair has a listed cost. Evaluate every availability and qualification\nrule stated by the query from the supplied eligibility fields, using the stated ordinal rank instead of alphabetical order. For each ineligible pair, exclude the pair or\nadd x[i,j] == 0; never infer eligibility from a nonblank cost. State the resulting pair mask\nbefore applying the one-to-one matching constraints.\n"""


def formulate_with_csvqa(query, dataset_address, route, prefix, suffix):
    """Retain route-specific ReAct modeling; the CSV tool itself never calls an LLM."""
    qa_tool, csvqa_result = build_csvqa_components(dataset_address)
    llm = make_llm()
    prefix += ("\nPreserve the original objective sense, full expression, and constant terms. "
               "Do not replace it with an optimizer-equivalent objective with a different value.")
    prefix += "\nPreserve the current query's conditional versus unconditional bounds. A fixed activation fee does not make a quantity lower bound conditional: apply an unconditional bound even when a category has no selected item. Use activation-conditioned bounds only if the query explicitly makes them conditional."
    prefix += QUERY_SEMANTICS_GUIDANCE
    prefix += "\nUse the ReAct protocol: after reasoning/tools, begin the final model with the literal marker Final Answer:. CSVQA always returns every input row and column without filtering. Complete source data is evidence, not automatic selection: apply only the original query's subset and eligibility rules in the model. Call CSVQA before the Final Answer; do not ask it to extract, compute, or summarize."
    agent = initialize_agent(
        tools=[qa_tool], llm=llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
        agent_kwargs={"prefix": prefix, "suffix": suffix}, verbose=True,
        handle_parsing_errors=True, return_intermediate_steps=True, early_stopping_method="generate",
    )
    result = invoke_react_with_required_csvqa(agent, query, route)
    return {"formulation": result["output"], **csvqa_result}


@lru_cache(maxsize=1)
def _load_rag_table():
    """Read the unified examples once, preserving every field as text."""
    frame = read_csv_compat(RAG_EXAMPLES_ALL_PATH, dtype=str, keep_default_na=False)
    required = {"prompt", "Data_address", "Related", "Required Data", "Label", "Code", "Type"}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"RAG example CSV is missing columns: {sorted(missing)}")
    frame["Type"] = frame["Type"].map(normalize_route)
    return frame


def load_rag_examples(route):
    """Return one canonical route; UFLP maps to FLP and Mixture to Others."""
    frame = _load_rag_table()
    examples = frame.loc[frame["Type"].eq(normalize_route(route))].copy()
    if examples.empty:
        raise ValueError(f"No RAG examples found for route {route!r}")
    return examples


@lru_cache(maxsize=None)
def _get_rag_store(route):
    """Share a semantic index across modeling and code generation for each route."""
    documents = [
        Document(
            page_content=example["prompt"],
            metadata={"example_json": json.dumps(example, ensure_ascii=False)},
        )
        for example in load_rag_examples(route).to_dict("records")
    ]
    # Embed the problem only: code/formulation length must not dominate similarity.
    return FAISS.from_documents(documents, make_embeddings())


def retrieve_rag_examples(route, query, k=1):
    """Retrieve structured example dictionaries, never parse CSVLoader display text."""
    if k < 1:
        return []
    route = normalize_route(route)
    documents = _get_rag_store(route).similarity_search(
        str(query), k=min(k, len(load_rag_examples(route))),
    )
    return [json.loads(document.metadata["example_json"]) for document in documents]


def retrieve_similar_texts(query, retriever):
    # The separate query-only reference dataset still uses its original retriever.
    return [doc.page_content for doc in retriever.invoke(query)]


def escape_braces(text):
    """Protect literal example braces when embedding text into an agent template."""
    return text.replace("{", "{{").replace("}", "}}")


def rag_example_observation(example):
    """Prefer the curated evidence, including its original source-row identifiers."""
    evidence = example["Required Data"].strip()
    if evidence:
        return evidence
    blocks = []
    for raw in example["Data_address"].splitlines():
        if raw.strip():
            path = Path(raw.strip()).expanduser()
            path = path if path.is_absolute() else PROJECT_ROOT / path
            frame = read_csv_compat(path, dtype=str, keep_default_na=False)
            blocks.append(f"Source: {raw.strip()}\n{frame.to_json(orient='records', force_ascii=False)}")
    return "\n\n".join(blocks)


def build_formulation_examples(route, query, k=1):
    """Render the common ReAct example format once for all CSV modeling routes."""
    examples = []
    for example in retrieve_rag_examples(route, query, k):
        block = f"""Question: Based on the following problem description and data, formulate a complete mathematical model. {example['prompt']}

Thought: Retrieve all data needed by the question, in source order. Preserve identifiers, coefficients, and any additional constraints. If the question does not request a subset, retrieve the whole dataset.

Action: CSVQA

Action Input: Retrieve all data {example['Related'].strip()} required to formulate this model. Preserve source order and exact identifiers and values, without simplification or abbreviation.

Observation: {rag_example_observation(example)}

Thought: Use the retrieved data as parameters and include all query-specific constraints. Choose variable domains from the question. Return the complete mathematical model in markdown, with $ or $$ for mathematical expressions and without extra explanations or LaTeX environments.

Final Answer:
{example['Label']}"""
        # Prefix is itself a PromptTemplate; escape the entire block exactly once.
        examples.append(escape_braces(block))
    return "\n\n".join(examples)


def get_NRM_response(query, dataset_address):
    few_shot_examples = build_formulation_examples("NRM", query, k=1)

    route_instruction = """Call CSVQA exactly once and return an ABSTRACT model. Define
index sets, parameters, variables, objective, and constraints symbolically. Include a short Data
Mapping that identifies every source by exact file path and exact column name. Do not copy record
values or literal record counts. CSVQA returns every row and column without selection.
State and implement query-supported subsets explicitly in Data Mapping; never assume that
returned rows have already been filtered, and never select by a guessed complete label."""
    prefix = f"""You are an assistant that formulates mathematical optimization models.

    The examples below illustrate model structure only; never copy their data.

    {few_shot_examples}

    {route_instruction}
    Use the user query as the authority for objective sense, variable domains, boundary semantics,
    and additional constraints. Do not invent missing data or silently omit returned data.
    """

    suffix = """

    Begin!

    User Description: {input}
    {agent_scratchpad}"""

    for attempt in range(2 if NRM_RETRY_ON_TRUNCATION else 1):
        try:
            result = formulate_with_csvqa(
                query, dataset_address, "NRM", prefix, suffix,
            )
            if attempt:
                result.setdefault("trace", {})["truncation_retry"] = {"llm_usage": truncated_usage}
            return result
        except ModelOutputTruncated as exc:
            if attempt or not NRM_RETRY_ON_TRUNCATION:
                raise
            truncated_usage = getattr(exc, "llm_usage", {})
            prefix += "\nReturn a concise symbolic model and Data Mapping only; never enumerate data rows or repeat examples."
            print("NRM formulation truncated; retrying once.", truncated_usage)


## 8. Resource allocation

Build the route-specific formulation using complete pandas-read CSV data and retrieved examples.


In [9]:
def get_RA_response(query, dataset_address):
    few_shot_examples = build_formulation_examples("RA", query, k=3)

    route_instruction = """You MUST call CSVQA exactly once before the Final Answer and inspect the complete returned data.
Apply only the original query's subset and eligibility rules; the reader has not selected rows.
Keep the supplied file order and original row order. Never sort or reset an index.
Use an explicit business ID column when present; never synthesize business IDs from row numbers.
Do not infer a file's semantic role only from its filename or position.
Align profits, resource consumption, and capacities through explicit column/ID keys.
Every resource-capacity dimension must receive a capacity constraint, and each
activity must appear in the objective and all applicable resource constraints.
Infer integer/binary/continuous domains from the user description and data.
An item count or number of units to order, produce, allocate, or develop is a nonnegative integer
unless the original query explicitly permits fractional quantities or describes divisible material.
The words scale, quantity or allocation alone do not establish continuous variables.
Distinguish a global activity decision from separate resource-by-activity allocations. When independent
warehouses, machines or other resources can each allocate to activities, use decisions indexed by
both resource and activity; do not constrain one identical global expression by every resource's limit.
Each resource-indexed capacity constraint must depend on that resource through its decisions or
consumption coefficients. For different resource dimensions consumed by one global activity, retain
global activity variables and the dimension-specific consumption coefficients instead.
Never invent a total capacity, default RHS, or omitted consumption coefficient.
Return a complete numerical formulation containing the retrieved identifiers and coefficients.
"""
    prefix = f"""You are an assistant that formulates mathematical optimization models.

    The examples below illustrate model structure only; never copy their data.

    {few_shot_examples}

    {route_instruction}
    Use the user query as the authority for objective sense, variable domains, boundary semantics,
    and additional constraints. Do not invent missing data or silently omit returned data.
    """

    suffix = """

    Begin!

    User Description: {input}
    {agent_scratchpad}"""

    return formulate_with_csvqa(
        query, dataset_address, "RA", prefix, suffix,
    )


## 9. Transportation

Build the route-specific formulation using complete pandas-read CSV data and retrieved examples.


In [10]:
def get_TP_response(query, dataset_address):
    few_shot_examples = build_formulation_examples("TP", query, k=3)

    route_instruction = """Call CSVQA at least once and return a complete numerical formulation. Retrieved
Information must contain every identifier and coefficient required by downstream code. Preserve
source order and do not invent, aggregate, or silently omit data."""
    prefix = f"""You are an assistant that formulates mathematical optimization models.

    The examples below illustrate model structure only; never copy their data.

    {few_shot_examples}

    {route_instruction}
    Use the user query as the authority for objective sense, variable domains, boundary semantics,
    and additional constraints. Do not invent missing data or silently omit returned data.
    """

    suffix = """

    Begin!

    User Description: {input}
    {agent_scratchpad}"""

    return formulate_with_csvqa(
        query, dataset_address, "TP", prefix, suffix,
    )


## 10. Assignment

Build the route-specific formulation using complete pandas-read CSV data and retrieved examples.


In [11]:
def get_AP_response(query, dataset_address):
    few_shot_examples = build_formulation_examples("AP", query, k=1)

    prefix = f"""You are an assistant that generates a mathematical model based on the user's description and provided CSV data.

            Please refer to the following example and generate the answer in the same format:

            {few_shot_examples}

            Note: Please retrieve all neccessary information from the CSV file to generate the answer. When you generate the answer, please output required parameters in a whole text, including all vectors and matrices.

            {AP_ELIGIBILITY_GUIDANCE}

            When you need to retrieve information from the CSV file, use the provided tool.

            """

    suffix = """

            Begin!

            User Description: {input}
            {agent_scratchpad}"""

    return formulate_with_csvqa(
        query, dataset_address, "AP", prefix, suffix,
    )


## 11. Facility location

Build the route-specific formulation using complete pandas-read CSV data and retrieved examples.


In [12]:
def get_FLP_response(query, dataset_address):
    few_shot_examples = build_formulation_examples("FLP", query, k=1)

    prefix = f"""You are an assistant that generates a mathematical model based on the user's description and provided CSV data.

            Please refer to the following example and generate the answer in the same format:

            {few_shot_examples}

            Note: Please retrieve all neccessary information from the CSV file to generate the answer. When you generate the answer, please output required parameters in a whole text, including all vectors and matrices.

            When you need to retrieve information from the CSV file, use the provided tool.

            """

    suffix = """
            Begin!

            User Description: {input}
            {agent_scratchpad}"""

    return formulate_with_csvqa(
        query, dataset_address, "FLP", prefix, suffix,
    )


## 12. Other and mixed problems

Keep the original Abstract Model Plan → code sequence and few-shot references.
The same complete pandas table-text Observation reaches both modeling and code
generation. Code generation also receives the entire abstract plan, query, input
file paths, and existing code examples. Query-derived guidance remains unchanged.


In [13]:
def build_few_shot_Other(examples, t="Model"):
    """Render already-structured examples; inserted prompt values need no brace escaping."""
    field = "Label" if t == "Model" else "Code"
    thought = (
        "I need to create an Abstract Model Plan based on the query and CSV schema."
        if t == "Model" else
        "I need to generate a single, complete, executable Gurobi Python code block."
    )
    return "\n\n".join(
        f"<EXAMPLE>\nQuestion: {example['prompt']}\nThought: {thought}\n"
        f"Final Answer:\n{example[field]}\n</EXAMPLE>"
        for example in examples if example[field].strip()
    )




SOURCE_TABLE_GUIDANCE = '\nIdentify logical tables from actual schema fields and table tags, never a guessed file index.\nA logical table may span several files: collect all its rows before query-directed selection,\njoining or aggregation. Apply a query-specified revision/as-of/deletion/retransmission rule to\nEVERY applicable table, before numeric conversion or sums. Compare revisions numerically,\nretain the latest eligible event by the full stated record key, then remove its tombstone.\nDo not convert blank fields in deleted or other-table rows into numeric parameters, and do\nnot silently coerce required malformed numbers to zero. Do not collapse distinct signed\ncomponents into a single last-write dictionary entry when the query requires their sum.\nKeep already consistent opaque business IDs as keys. Use a lookup only when crossing key\nnamespaces; a display label is not a replacement for a working canonical key.\nConvert all quantities in a resource comparison to the same stated unit, on both sides.\n'


def get_Others_response(user_query: str, dataset_address: str):
    examples = retrieve_rag_examples("Others", user_query, k=3)
    llm2 = make_llm()

    try:
        print("\n[Gurobi Pipeline Step 1/3]: Reading complete CSV data with pandas...")

        csv_result = load_csv_data_result(dataset_address)
        query_guidance = ""
        if re.search(r"\bsymmetric\b", user_query, re.I):
            query_guidance += ("\nWhen the query explicitly states symmetry, align matrix axes by entity ID. "
                       "Fill a missing off-diagonal entry from its present transpose; preserve zeros. "
                       "Reject pairs with both entries missing or conflicting values beyond numeric tolerance. "
                       "Without explicit symmetry, preserve direction and never mirror or zero-fill entries.")
        if re.search(r"\bshut(?:down| down)\b", user_query, re.I):
            query_guidance += "\nMinimum-down-time restrictions activate only on an on-to-off transition; down-time constraints alone must never forbid an always-on sequence."
        if "\\sum" in user_query:
            query_guidance += "\nPreserve the query's explicit summation scope: aggregate every summed index inside the expression; never replace a summed index with a separate family of constraints."
        if re.search(r"\bworkers?\b", user_query, re.I) and re.search(r"\btasks?\b", user_query, re.I):
            query_guidance += "\nIn a worker-task matrix, distinguish embedded row/column-axis captions from actual workers. Select workers by their identifiers, exclude label rows, then validate worker counts; never truncate by position."
        schema = csv_result["observation"] + query_guidance
        print("\n[Gurobi Pipeline Step 2/3]: Constructing Abstract Model...")

        few_shot_block_abstract = build_few_shot_Other(examples, t='Model')
        print(f"[Few-Shot Abstract Examples]:\n{few_shot_block_abstract}...")
        
        abstract_model_template = """
You are an expert optimization modeler.
Your task is to create an "Abstract Model Plan" based on the user's query and the complete CSV data read with pandas.
This plan is *not* Gurobi code or mathematical formulas, but a clear, step-by-step reasoning process in English.

[Examples]
{few_shot_examples}

[Current Task]
User Query: {query}

Complete CSV Data:
{schema}

An explicitly enumerated complete entity set in the query is authoritative: extra CSV entities
must not enlarge that set unless the query explicitly delegates its definition to the data.
For a one-off finite-horizon schedule, change/ramp constraints compare consecutive modeled periods. An initial
binary on/off state is not a pre-horizon continuous quantity: do not invent an initial continuous
level or a first-period ramp limit unless the query explicitly supplies that continuous boundary.
For recurring daily/cyclic operation, preserve wrap-around coverage across the last/first period
when a working shift spans the boundary; do not impose a fictitious empty start to a recurring day.
State and implement the original objective sense and complete value expression, including constants.
Do not offer sign-reversed or constant-shifted optimizer-equivalent objectives as alternatives.
Examples illustrate style; the current query and data determine the actual scope, indices and boundaries.

Data selection: state for each entity table whether all rows or a subset is required; name the
filter column, exact condition, entity key, and joins. Preserve shared resource/parameter tables.
Without a query-requested filter, use the complete entity set. All CSV rows are supplied as
source evidence, not as an automatically selected set. Apply only query-supported predicates;
do not broaden a zero-match condition to substring matching or all rows. Check selected keys
and applicable counts at runtime.

[Your Output]
You must strictly follow this format for your "Abstract Model Plan" output:

[Abstract Model Plan START]
1.  **Analyze Query:** The user wants to {{{{Your analysis}}}}.
2.  **Identify Model Type:** Based on the query, this is a {{{{e.g., LP, MIP, Fixed-Charge, Blending}}}} problem.
3.  **Define Index Sets:** The primary indices are {{{{e.g., Products, Workers, Sources, Destinations}}}}.
4.  **Define Decision Variables:**
    -   `x[i]` = {{{{Describe first variable, e.g., 'quantity of product i'}}}}. Type: {{{{GRB.CONTINUOUS / GRB.INTEGER}}}}.
    -   `y[i]` = {{{{Describe second variable, e.g., 'if product i is produced'}}}}. Type: {{{{GRB.BINARY}}}}.
5.  **Identify Parameters (from Schema):**
    -   Objective coefficients (e.g., profit) will come from column(s): {{{{e.g., 'Price', 'Production Cost'}}}}.
    -   Constraint coefficients (e.g., resource use) will come from: {{{{e.g., 'Resource 1', 'Available Time'}}}}.
    -   Constraint RHS (limits) will come from: {{{{e.g., 'Max Demand'}}}}.
6.  **Formulate Objective:** {{{{Describe objective, e.g., 'Maximize sum((schema['Price'] - schema['Cost']) * x[i] - schema['FixedCost'] * y[i])'}}}}.
7.  **Formulate Constraints:**
    -   Constraint 1 (e.g., Resource Limit): {{{{Describe constraint 1, e.g., 'sum(schema['Resource 1'][i] * x[i]) <= schema['Available']'}}}}.
    -   Constraint 2 (e.g., Linking): {{{{Describe constraint 2, e.g., 'x[i] <= M * y[i])'}}}}.
    -   ... (Other constraints) ...
[Abstract Model Plan END]
"""
        abstract_model_template += "\nPreserve the current query's conditional versus unconditional bounds. A fixed activation fee does not make a quantity lower bound conditional: apply an unconditional bound even when a category has no selected item. Use activation-conditioned bounds only if the query explicitly makes them conditional.\nReturn exactly one concise abstract plan with the seven requested sections. Describe each constraint family once. Use symbolic index sets and exact field mappings; do not enumerate source rows, repeat examples, or emit code. Preserve every required variable, domain, objective term and boundary condition."
        abstract_model_template += QUERY_SEMANTICS_GUIDANCE + SOURCE_TABLE_GUIDANCE
        abstract_prompt = PromptTemplate(
            template=abstract_model_template,
            input_variables=["query", "schema", "few_shot_examples"]
        )
        
        abstract_model_chain = LLMChain(llm=llm2, prompt=abstract_prompt)
        
        abstract_model_plan = abstract_model_chain.run(
            query=user_query,
            schema=schema,
            few_shot_examples=few_shot_block_abstract
        )
        print(f"[Observation]:\n{abstract_model_plan}")
        
        print("\n[Gurobi Pipeline Step 3/3]: Generating Gurobi Code...")
        
        few_shot_block_code = build_few_shot_Other(examples, t='Code')
        
        code_gen_template = """
You are an expert Gurobi programmer.
Your task is to strictly follow the User Query, CSV Observation, and "Abstract Model Plan" to translate them into a single, complete, executable Gurobi Python code block.
The code must start with ```python and end with ```.
The code must include the necessary imports from gurobipy, pandas, numpy, re, math or json. Do not import sys or os.
Create exactly one Gurobi model in m and call its optimize() exactly once, both at top level. Do not wrap model creation or solving in functions or try/except; let errors propagate to the execution harness. Do not write files or call computeIIS().
The code must read all supplied CSV files needed by the model, at these exact newline-separated paths:
{dataset_address}
Read their complete contents at runtime.
Preserve explicit business identifiers and matrix axes; do not invent filenames or columns.
Data-reading contract: these files were parsed with pandas.read_csv(path, sep=',').
Read the original CSV files with a comma delimiter.
pandas already consumes the CSV header. Select entity and parameter rows through identifier values
or validated predicates, never guessed iloc offsets or a guessed number of header rows.
Normalize identifier types explicitly on both sides of every lookup/join, including matrix row
indices versus column labels. Preserve the actual identifiers and derive dimensions from all rows.
Series.apply receives one cell value, not a row; use vectorized column predicates or DataFrame.apply
with axis=1 when a predicate needs a row. Every comprehension must use the indices it binds, not
stale variables from a preceding loop. Do not silently skip failed conversions or missing coefficients.
Use Series.str.strip().str.casefold() for vectorized string normalization, or apply a scalar function;
Series has no casefold() method. For arbitrary CSV headers (spaces, punctuation, numeric names),
use row[column] or explicit arrays, not getattr on itertuples(), which renames invalid field names.
Series.to_dict() uses the DataFrame index, not an ID column: set the validated entity ID as index
or construct dictionaries with explicit ID/value pairs. Preserve identifier case by default. If
normalization is required, apply the same mapping to entity values and matrix column labels.
When a supplied identity/lookup table relates opaque references to entity IDs, resolve foreign
keys through that table and validate coverage; never compare an opaque reference directly to
a display label as if they were the same key.
Use setObjective for the exact metric requested by the query, including all additive constants,
signs and units. An optimizer-equivalent objective is not an equivalent reported objective value;
do not drop constant terms and then merely restore them in printed output.
The code must *fully* implement all variables, objectives, and constraints from the "Abstract Model Plan".

[Examples of the Full Process]
{few_shot_examples}

[User Query]
{query}

[CSV Observation (pandas table text)]
{observation}

[Abstract Model Plan]
{abstract_plan}

An explicitly enumerated complete entity set in the query is authoritative: extra CSV entities
must not enlarge that set unless the query explicitly delegates its definition to the data.
For a one-off finite-horizon schedule, change/ramp constraints compare consecutive modeled periods. An initial
binary on/off state is not a pre-horizon continuous quantity: do not invent an initial continuous
level or a first-period ramp limit unless the query explicitly supplies that continuous boundary.
For recurring daily/cyclic operation, preserve wrap-around coverage across the last/first period
when a working shift spans the boundary; do not impose a fictitious empty start to a recurring day.
State and implement the original objective sense and complete value expression, including constants.
Do not offer sign-reversed or constant-shifted optimizer-equivalent objectives as alternatives.
Examples illustrate style; the current query and plan determine the actual scope, indices and boundaries.

Data selection: state for each entity table whether all rows or a subset is required; name the
filter column, exact condition, entity key, and joins. Preserve shared resource/parameter tables.
Without a query-requested filter, use the complete entity set. Read the source rows at runtime
and apply only query-supported predicates;
do not broaden a zero-match condition to substring matching or all rows. Check selected keys
and applicable counts at runtime.

Reserve m and model for the Gurobi model; never reuse them as loop variables or other values.
Keep business identifiers as Python dictionary keys, separate from Gurobi names.
Use name='' for addVars/addConstrs so business keys are not expanded into solver names;
use only short ASCII names for individually named variables and constraints.
Never invent, truncate, or replace required coefficients with dummy or random data.
Before optimization, validate coefficient dimensions and identifier coverage against decision index sets;
raise an explicit error for missing data instead of guessing, padding, or silently dropping entries.

[Your Gurobi Code]
"""

        code_gen_template += "\nGive each decision-variable container a distinct name ending in _vars; never reuse it as a loop variable, scalar key, parameter or row object. In a generator expression, index the decision container with the locally bound key, not with a shadowed container. quicksum takes an iterable; a scalar constant is used directly.\nRead source CSVs with dtype=str and keep_default_na=False, then explicitly convert the required numeric fields. Preserve blank forbidden entries and exact identifiers.\nPreserve the current query's conditional versus unconditional bounds. A fixed activation fee does not make a quantity lower bound conditional: apply an unconditional bound even when a category has no selected item. Use activation-conditioned bounds only if the query explicitly makes them conditional.\n\nMandatory result interface: expose exactly one solved, live Gurobi model at module scope as m. Create and solve m at top level as required above; keep it live until result extraction. Never assign optimize()'s return to m. Do not dispose the model or leave its context manager before the harness extracts the result. Printed objectives cannot replace this model object.\n"
        code_gen_template += QUERY_SEMANTICS_GUIDANCE + SOURCE_TABLE_GUIDANCE + query_guidance
        code_gen_template += "\nDataFrame.columns is an Index, not a dictionary: test column_name in df.columns, then access df[column_name]. Do not call columns.get(). Gurobi comparisons are TempConstr objects: pass them to addConstr, never multiply them or pass them to quicksum. For a positive-quantity prerequisite, use the already linked binary selection flags: selected_item <= selected_prerequisite, without a quantity ratio."
        code_gen_prompt = PromptTemplate(
            template=code_gen_template,
            input_variables=["query", "observation", "abstract_plan", "dataset_address", "few_shot_examples"]
        )
        
        code_gen_chain = LLMChain(llm=llm2, prompt=code_gen_prompt)
        
        final_code = code_gen_chain.run(
            query=user_query,
            observation=csv_result["observation"],
            abstract_plan=abstract_model_plan,
            dataset_address=dataset_address,
            few_shot_examples=few_shot_block_code 
        )
        
        if "```python" in final_code:
            code_block = final_code.split("```python", 1)[1]
            if "```" in code_block:
                code_block = code_block.split("```", 1)[0]
            final_answer = "```python\n" + code_block.strip() + "\n```"
        else:
            if not final_code.strip().startswith("import"):
                print(f"[Warning] Step 3 output was not a valid code block. Output: {final_code[:200]}...")
                final_answer = f"Error: Code generation failed. LLM returned non-code output:\n{final_code}"
            else:
                print("[Warning] Step 3 output missed ```python tag, adding it.")
                final_answer = "```python\n" + final_code.strip() + "\n```"
            
        print(f"[Final Answer]:\n{final_answer}")
        return {"formulation": abstract_model_plan, "code": final_answer, **csv_result}

    except Exception as e:
        print(f"[Gurobi Pipeline Error]: {e}")
        raise


def get_others_without_CSV_response(query):
    llm = make_llm()

    loader = CSVLoader(file_path=str(PROJECT_ROOT / "Large_Scale_Or_Files/RAG_Example_Others_Without_CSV.csv"), encoding="utf-8")
    documents = loader.load()

    embeddings = make_embeddings()
    vectors = FAISS.from_documents(documents, embeddings)

    retriever = vectors.as_retriever(search_kwargs={'k': 5})
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
    )

    qa_tool = Tool(
        name="ORLM_QA",
        func=qa_chain.invoke,
        description=(
            "Use this tool to answer Querys."
            "Provide the Query as input, and the tool will retrieve the relevant information from the file and use it to answer the Query."
        ),
    )

    few_shot_examples = []
    similar_results = retrieve_similar_texts(query, retriever)

    for content in similar_results:

        split_at_formulation = content.split("Data_address:", 1)
        problem_description = split_at_formulation[0].replace("prompt:", "").strip()
        split_at_address = split_at_formulation[1].split("Label:", 1)
        split_at_label = split_at_address[1].split("Related:", 1)
        label = split_at_label[0].strip() 

        example = (
            "<EXAMPLE>\n"
            f"Question: {problem_description}\n\n"
            "Thought: Read the question and 1) identify the goal (minimize time/cost/crew or maximize throughput/value) and collect per-unit coefficients; 2) define decision variables and pick domains: counts of trips/units/vehicles are nonnegative integers, yes or no choices are binary, divisible flows or weights are nonnegative reals; 3) write the linear objective from the coefficients; 4) add constraints in this order: demand or target (exactly, at least, at most), capacity or supply upper bounds, flow conservation or stage linking across nodes or arcs, and any share or ratio limits rewritten as linear inequalities using the given per-unit rates, plus any minimum or maximum usage; 5) add nonnegativity and the chosen integrality or binary domains; 6) output only the LP: objective first, then constraints line by line with brief labels if needed, then a final line stating the variable domains."
            "Final Answer:\n"
            f"{label}\n"
            "</EXAMPLE>"
        )

        example = example.replace("{", "{{").replace("}", "}}")
        few_shot_examples.append(example)

    prefix = (
    f"{few_shot_examples}\n\n"
    """ 
    Use the following triggers to identify problem structures and apply the corresponding mathematical formulations.
    (1) INTEGER trigger:
    Question: A factory can run two machine types. How many of each machine should be installed given budget and space limits? Maximize output.
    Final Answer: 
    $\\max\\; p_1 x_1 + p_2 x_2$
    $\\text{{s.t. }} a_1 x_1 + a_2 x_2 \\le B,\\; s_1 x_1 + s_2 x_2 \\le S$
    $x_1, x_2 \\in \\mathbb{{Z}}_+.$
    (2) MULTI-PERIOD FLOW trigger:
    Question: Multi-period production with inventory and backorders. Costs for production/holding/backorder. Initial and terminal conditions given.
    Final Answer:
    \textbf{{Indices: }} t\\in T=\\{{1,\\dots,n\\}}. \\
    \textbf{{Given: }} d_t,\\ I_0,\\ B_0,\\ \\dots \\
    \textbf{{Vars: }} x_t\\ge0,\\ I_t\\ge0,\\ B_t\\ge0. \\
    \\min \\sum_t (c x_t + h I_t + p B_t) \\
    \text{{s.t. }} I_t - B_t = I_{{t-1}} - B_{{t-1}} + x_t - d_t,\\ \forall t \\
    I_n \\ge I^{{\\min}},\\ B_n=0.
    (3) LOGIC+BINARY trigger:
    Question: Choose exactly one option from set P and at least K items from set V, with quantities and budget.
    Final Answer:
    \textbf{{Sets: }} i\\in P,\\ j\\in V. \\ \textbf{{Vars: }} q_i,q_j\\ge0;\\ y_i\\in\\{{0,1\\}}, z_j\\in\\{{0,1\\}}.\\
    \\max \\sum_j f_j q_j \\
    \text{{s.t. }} \\sum_i y_i = 1,\\ \\sum_j z_j \\ge K \\
    0\\le q_i \\le M_i y_i,\\ \forall i;\\ \\ 0\\le q_j \\le M_j z_j,\\ \forall j \\
    \text{{Budget: }} \\sum_i c_i q_i + \\sum_j c_j q_j \\le B.
    """
    "USER QUESTION:\n{input}\n\n"
    "TASK:\n"
    "- Produce a complete LaTeX optimization model using ONLY information in the question.\n"
    "- Use a fixed structure INSIDE LaTeX: Indices/Sets; Given Parameters (convert all tables to arrays); Decision Variables (with domains); Objective; Constraints; Domain lines.\n"
    "- For multi-period problems, you MUST include state-balance recurrences and initial/terminal conditions explicitly.\n"
    "- Avoid nonlinear forms when possible: rewrite ratios/logic using linear constraints + binaries (big-M) with clearly defined M.\n\n"
    "You should decide VARIABLE–TYPE first!\n (Use integers for indivisible item counts, binaries for choices, and continuous variables for explicitly divisible flows or material.)"
    "### FIRST RESPONSE FORMAT (exactly 3 lines) ###\n"
    "Thought: <brief>\n"
    "Action: ORLM_QA\n"
    "Action Input: {input}\n\n"
    "Begin."
)

    suffix = (
    "\n### AFTER OBSERVATION ###\n"
    "Respond with exactly two lines:\n"
    "Thought: <variable types: integer/binary/continuous>\n"
    "Final Answer: <ONLY LaTeX model. Must include: (i) indices/sets, (ii) parameter definitions (tables->arrays), (iii) explicit domain lines for EVERY variable, (iv) initial/terminal conditions if any. No prose.>\n"
    "Do NOT output anything else."
)

    agent = initialize_agent(
        tools=[qa_tool],
        llm=llm,
        agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
        agent_kwargs={
            "prefix": prefix,
            "suffix": suffix,
            "input_variables": ["input"]
        },
        verbose=True,
        handle_parsing_errors=True,  # Enable error handling
    )

    openai.api_request_timeout = 60  
    output = agent.run({"input": query})

    return output


## 13. Generate Python code

The shared code prompt receives the full unchanged formulation, original query,
existing code examples, original CSV paths, and the exact pandas table-text Observation
already used for modeling. It does not reread headers or records, filter the model,
parse the Observation into JSON, or prepend preloaded data to the generated program.
Query-only code generation is unchanged. Generated CSV programs still read source files.


In [14]:
# ============================================================
# Code generation: shared RAG examples and one prompt builder
# ============================================================

ORIGINAL_CODE_PROMPT = """
You are an expert in mathematical optimization and Python programming.
Write executable Python code that solves the provided mathematical optimization
model with Gurobi, including every decision variable, objective, and constraint.
Preserve variable domains consistent with the original query; the original query takes
precedence over conflicting intermediate wording. Return code only, without explanations
or Markdown fences.

Mathematical Optimization Model:
{output}
"""

CSV_ROUTE_HINT = {
    "FLP": "Preserve facility-opening and assignment indices and their linking constraints.",
    "AP": "Preserve assignment indices and the matching constraints in the formulation.\n" + AP_ELIGIBILITY_GUIDANCE.strip(),
    "TP": "Preserve source-destination axes and supply-demand balance; never assume a square matrix.",
    "RA": """Preserve resource dimensions and capacity constraints. Keep independent
resource-by-activity allocations distinct from global activities consuming multiple
resource dimensions. Index independent allocations by resource, and use each
resource's own decisions or consumption coefficients in its capacity constraint.
Never reuse a global allocation expression against every independent warehouse's
capacity. Preserve all supplied capacities, coefficients, and business identifiers.
Item counts and discrete units must be nonnegative integers unless the problem
explicitly allows fractional quantities or describes divisible material; do not
infer continuity from scale alone.""",
    "NRM": "Preserve product/resource indices and dimension-specific capacity coefficients.",
    "Others": "Follow the formulation exactly; do not introduce unsupported assumptions.",
}

CSV_FILE_CODE_INSTRUCTIONS = """
Use the complete formulation and Original Query for model structure and data mapping.
Read required data from the original CSV files at the exact paths listed below.
No preloaded data variables are provided; initialize all data used by the program.
"""

QUERY_ONLY_CODE_INSTRUCTIONS = """
Build a self-contained program using the concrete coefficients and identifiers in
the Mathematical Optimization Model / Retrieved Information and Original Query.
Define all required data in the code. Do not use CSVQA_DATA, external variables,
or external files.
"""

CSV_SOLVER_INSTRUCTIONS = """
Keep business identifiers as Python dictionary keys, separate from Gurobi names.
Use name='' for addVars/addConstrs so business keys are not expanded into solver names;
use only short ASCII names for individually named variables and constraints.
Use setObjective for the original query's objective sense and full expression, including constants.
Do not substitute an optimizer-equivalent objective or restore omitted constants only in printed output.
Never invent, truncate, or replace required coefficients with dummy or random data.
Before optimization, validate coefficient dimensions and identifier coverage against decision index sets;
raise an explicit error for missing data instead of guessing, padding, or silently dropping entries.
For |linear expression| <= bound, use expression <= bound and expression >= -bound;
do not pass a non-variable expression to addGenConstrAbs.
Leave the solved Gurobi model in m. If using a function, return the model
and assign it to m. Set MIPGap=1e-4 before optimize().
If Status == GRB.OPTIMAL, print ObjVal and every variable's VarName and X.
Otherwise print the solver status; do not report an incumbent as optimal.
"""

CSV_SOLVER_INSTRUCTIONS += QUERY_SEMANTICS_GUIDANCE


def retrieve_csv_code_example(route, query):
    """Render code references from the same structured examples used for modeling."""
    examples = []
    for row in retrieve_rag_examples(normalize_route(route), query, k=2):
        if row["Code"].strip():
            examples.append(
                f"Example {len(examples) + 1}:\n"
                f"Problem:\n{row['prompt']}\n\n"
                f"Formulation:\n{row['Label']}\n\n"
                f"Code:\n{row['Code']}"
            )
    return "\n\n".join(examples)


def _generate_code(output, route, original_query="", dataset_address="", observation=""):
    """Pass the full model and existing pandas Observation unchanged; do not reread CSVs."""
    route = normalize_route(route)
    parts = [ORIGINAL_CODE_PROMPT.format(output=output)]
    if original_query:
        parts.append(f"Original Query:\n{original_query}")
    if observation:
        parts.append(f"Complete CSV Observation (pandas table text):\n{observation}")
    if dataset_address:
        parts.extend([CSV_FILE_CODE_INSTRUCTIONS,
                      f"Original CSV paths (source order):\n{resolve_dataset_address(dataset_address)}"])
        # Retain the baseline's non-data instructions from its respective data branches.
        if route == "NRM":
            parts.append(
                "Without a continuous pre-horizon value, start change/ramp constraints at the second\n"
                "period; an initial binary state does not supply a continuous period-zero value.\n"
                "Parse structured CSV text with a regex and fail explicitly if any clause is unmatched."
            )
        else:
            parts.append(
                "Derive dimensions and identifiers from the source CSV records. Original query determines "
                "semantics and domains; the formulation supplies structure, not guessed data. "
                "Keep explicit per-type capacity bounds; if total capacity denotes their aggregate, "
                "compute their sum rather than inventing another shared limit."
            )
    else:
        parts.append(QUERY_ONLY_CODE_INSTRUCTIONS)
    parts.append(CSV_ROUTE_HINT[route])

    examples = retrieve_csv_code_example(route, original_query or output)
    if examples:
        parts.extend([
            "The following examples are coding references only. Reuse indexing and "
            "Gurobi modeling patterns, but never copy their coefficients, dimensions, "
            "identifiers, or data. The current formulation and current data are authoritative.",
            examples,
        ])
    parts.append(CSV_SOLVER_INSTRUCTIONS)
    parts.append("\nGive each decision-variable container a distinct name ending in _vars; never reuse it as a loop variable, scalar key, parameter or row object. In a generator expression, index the decision container with the locally bound key, not with a shadowed container. quicksum takes an iterable; a scalar constant is used directly.\nWhen the task permits CSV reading, use dtype=str and keep_default_na=False, then explicitly convert the required numeric fields; otherwise use only the supplied records or formulation data. Preserve blank forbidden entries and exact identifiers.\nPreserve the current query's conditional versus unconditional bounds. A fixed activation fee does not make a quantity lower bound conditional: apply an unconditional bound even when a category has no selected item. Use activation-conditioned bounds only if the query explicitly makes them conditional.\n\nMandatory result interface: expose exactly one solved, live Gurobi model at module scope as m. A function that creates a model must explicitly return m, and its caller must assign that result to m. Never assign optimize()'s return to m. Do not dispose the model or leave its context manager before the harness extracts the result. Printed objectives cannot replace this model object.\n")
    response = make_llm().invoke([HumanMessage(content="\n\n".join(parts))])
    code = response.content
    print(code)
    return code


def get_csv_code(output, route, original_query, dataset_address, observation=""):
    """Generate from the full model/query/examples, input paths, and pandas Observation."""
    return _generate_code(output, route, original_query, dataset_address=dataset_address,
                          observation=observation)


def get_code(output, selected_problem, original_query=""):
    """Generate self-contained code for a query-only formulation."""
    return _generate_code(output, selected_problem, original_query=original_query)


## 14. Execute and assemble results

Use frozen baseline predictions when supplied. The complete CSV Observation is saved
for audit and passed to code generation; it is not prepended to the generated program.
Others passes its Observation inside its own two-stage chain. The solver execution,
result extraction and error handling are unchanged from v4.
Execution remains in-process, not a security sandbox.


In [15]:
def _source_candidate(code):
    source = str(code).strip()
    match = re.fullmatch(r"```(?:python|py)?\s*\n(.*?)\n```", source, re.DOTALL)
    source = match.group(1) if match else source
    tree = ast.parse(source)
    # Bulk names expand business keys; keep those keys out of Gurobi display names.
    for node in ast.walk(tree):
        if (isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute)
                and node.func.attr in {"addVars", "addConstrs"}):
            node.keywords = [kw for kw in node.keywords if kw.arg != "name"]
            node.keywords.append(ast.keyword(arg="name", value=ast.Constant(value="")))
    return ast.unparse(tree)


def execute_code(code):
    namespace = {"__name__": "__main__"}
    exec(_source_candidate(code), namespace)
    model = namespace["m"] if "m" in namespace else namespace["model"]
    with model:
        if model.Status != gp.GRB.OPTIMAL:
            raise RuntimeError(f"No optimal solution; solver status: {model.Status}")
        objective = model.ObjVal
        solution = [(v.VarName, v.X) for v in model.getVars()]
        print("Optimal objective:", objective)
        print("Optimal solution:", solution)
        return objective, solution


ROUTE_FORMULATORS = {
    "NRM": get_NRM_response, "RA": get_RA_response, "TP": get_TP_response,
    "AP": get_AP_response, "FLP": get_FLP_response,
}


def invoke_formulation(route, query, dataset_address):
    if not dataset_address:
        return {"formulation": get_others_without_CSV_response(query), "trace": {"status": "NOT_APPLICABLE"}}
    return ROUTE_FORMULATORS.get(normalize_route(route), get_Others_response)(query, dataset_address)


def case_fields(case):
    reference = case.get("label_objective")
    address = normalize_data_address(case.get("dataset_address"))
    return {
        "problem_id": str(case["problem_id"]), "query": str(case["Query"]),
        "experiment_method": EXPERIMENT_METHOD,
        "dataset_address": address, "input_kind": "external_csv" if address else "query_only",
        "true_label": case.get("true_label"), "true_route": case.get("true_route"),
        "label_objective": float(reference) if reference is not None and pd.notna(reference) else None,
    }


def execute_pipeline_case(case, *, forced_route=None):
    """Classify, formulate, generate code, and solve; retain progress if a step fails."""
    record = {**case_fields(case), "forced_route": forced_route,
              "experiment_mode": "forced" if forced_route else "automatic"}
    HTTP_RETRY_EVENTS.clear()
    query, address = record["query"], record["dataset_address"]
    try:
        if forced_route is not None and not address:
            raise ValueError("Forced routes require CSV inputs")
        record["pipeline_stage"] = "classification" if forced_route is None else "formulation"
        classification = classification_for_case(case) if forced_route is None else {}
        label = classification.get("normalized_label")
        route = normalize_route(forced_route) if forced_route is not None else class_to_workflow_route(label)
        route = route if address else "Others"
        record.update(predicted_label=label, assigned_route=route,
                      classification_source=classification.get("classification_source", "forced_route"))
        record["pipeline_stage"] = "formulation"
        formulation = invoke_formulation(route, query, address)
        mode = CSVQA_MODE if address else None
        record.update(generated_model=formulation["formulation"], csvqa_observation=formulation.get("observation", ""), csvqa_mode=mode,
                      csvqa_status=formulation.get("trace", {}).get("status"))
        record["csvqa_trace"] = json.dumps(formulation.get("trace", {}), ensure_ascii=False)
        model_text = formulation["formulation"]
        if (not isinstance(model_text, str) or not model_text.strip()
                or "agent stopped due to" in model_text.lower()):
            raise RuntimeError("Invalid formulation: empty model or agent stopped before producing a model")
        record["pipeline_stage"] = "code_generation"
        code = formulation.get("code")
        if code is None:
            if address:
                code = get_csv_code(
                    formulation["formulation"], route, query, dataset_address=address,
                    observation=formulation.get("observation", ""),
                )
            else:
                code = get_code(formulation["formulation"], route, original_query=query)
        source = _source_candidate(code)
        record["solve_code"] = source
        record["pipeline_stage"] = "solve"
        objective, solution = execute_code(record["solve_code"])
        record.update(api_retry_count=len(HTTP_RETRY_EVENTS), api_retry_events=json.dumps(HTTP_RETRY_EVENTS))
        record.update(final_objective=objective, final_solution=solution, final_ok=True,
                      record_status="completed", cache_source="computed", pipeline_stage="completed")
        return record
    except BaseException as exc:
        record.update(api_retry_count=len(HTTP_RETRY_EVENTS), api_retry_events=json.dumps(HTTP_RETRY_EVENTS))
        exc.pipeline_context = record
        raise


## 15. Test datasets and scoring

Query is required. Dataset_address, Problem Type, and the reference objective (Label-objective or Label) are optional. Missing CSV paths select query-only modeling; missing reference labels leave the corresponding accuracy blank.


In [16]:
def normalize_data_address(value):
    if value is None or (not isinstance(value, str) and pd.isna(value)):
        return ""
    return "\n".join(line.strip() for line in str(value).splitlines() if line.strip())


def resolve_dataset_address(value):
    return "\n".join(
        str((path if path.is_absolute() else PROJECT_ROOT / path).resolve())
        for path in (Path(line).expanduser() for line in normalize_data_address(value).splitlines())
    )


def parse_label_objective(value):
    text = str(value).replace(",", "").replace("$", "").strip()
    match = re.search(r"Optimal\s+objective\s*([-+\d.eE]+)", text, re.I)
    result = float(match.group(1) if match else text)
    if not math.isfinite(result):
        raise ValueError(f"Reference objective is not finite: {value!r}")
    return result


def prepare_cases(test):
    """Accept Large-scale-or, IndustryOR, or a DataFrame with at least Query."""
    frame = test.copy()
    if "Query" not in frame:
        raise ValueError("The test dataset must contain a Query column")
    if frame["Query"].isna().any() or frame["Query"].astype(str).str.strip().eq("").any():
        raise ValueError("Every test row must contain a non-empty Query")
    if "problem_id" not in frame:
        frame["problem_id"] = [f"OR-{index + 1:03d}" if isinstance(index, (int, np.integer)) else f"OR-{position:03d}"
                               for position, index in enumerate(frame.index, 1)]
    if frame.empty or frame["problem_id"].isna().any():
        raise ValueError("Cases require nonempty IDs")
    frame["problem_id"] = frame["problem_id"].astype(str)
    if frame["problem_id"].str.strip().eq("").any() or frame["problem_id"].duplicated().any():
        raise ValueError("Cases require unique nonempty IDs")
    addresses = frame.get("dataset_address", frame.get("Dataset_address", pd.Series("", index=frame.index)))
    frame["dataset_address"] = addresses.map(resolve_dataset_address)
    labels = frame.get("true_label", frame.get("Problem Type", pd.Series(None, index=frame.index, dtype=object)))
    frame["true_label"] = labels.map(lambda value: normalize_problem_class(value) if pd.notna(value) and str(value).strip() else None)
    frame["true_route"] = frame["true_label"].map(lambda label: class_to_workflow_route(label) if label else None)
    objectives = frame.get("label_objective", frame.get("Label-objective", frame.get("Label", pd.Series(None, index=frame.index, dtype=object))))
    frame["label_objective"] = objectives.map(lambda value: parse_label_objective(value) if pd.notna(value) and str(value).strip() else None)
    return frame


def load_benchmark(path=BENCHMARK_PATH):
    return prepare_cases(read_csv_compat(path))


def objective_is_correct(actual, expected, *, rel_tol=1e-4, abs_tol=1e-4):
    try:
        actual, expected = float(actual), float(expected)
        return math.isfinite(actual) and math.isfinite(expected) and math.isclose(actual, expected, rel_tol=rel_tol, abs_tol=abs_tol)
    except (TypeError, ValueError, OverflowError):
        return False


def finalize_record(record, *, rel_tol=1e-4, abs_tol=1e-4):
    result = dict(record)
    result["classification_correct"] = (
        result.get("predicted_label") == result["true_label"]
        if result.get("true_label") and result.get("experiment_mode") == "automatic" else None
    )
    reference = result.get("label_objective")
    result["solution_correct"] = (
        bool(result.get("final_ok") and objective_is_correct(result.get("final_objective"), reference, rel_tol=rel_tol, abs_tol=abs_tol))
        if reference is not None and pd.notna(reference) else None
    )
    return result

## 16. Save and reuse results

Use `outputs/few_shot_only_v4`, separate from both the full baseline and previous
Few-shot variants. Keep model, code, solution, Observation and trace artifacts.
The new method/cache identity prevents reuse of earlier variants' solve artifacts.
Classification predictions remain part of automatic-run fingerprints.


In [17]:
def _write_text_atomic(path, text, encoding):
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", encoding=encoding, dir=path.parent, prefix=f".{path.name}.",
            suffix=".tmp", delete=False,
        ) as handle:
            handle.write(text)
            handle.flush()
            os.fsync(handle.fileno())
            temporary = Path(handle.name)
        temporary.replace(path)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)


CSV_VALUE_READERS = {
    **dict.fromkeys(("final_ok", "classification_correct", "solution_correct", "external_deadline_exceeded", "route_allowed"), lambda value: value == "True"),
    **dict.fromkeys(("final_objective", "label_objective"), float),
}
_RESULT_LOCK = RLock()


def load_records(csv_path):
    path = Path(csv_path)
    if not path.exists():
        return []
    rows = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig").to_dict("records")
    return [{key: CSV_VALUE_READERS.get(key, str)(value) if value else None
             for key, value in row.items()} for row in rows]


def record_key(record):
    return (str(record["problem_id"]), record.get("query", ""),
            normalize_data_address(record.get("dataset_address")), record.get("forced_route") or "AUTO")


RECORD_ARTIFACT_FILES = {
    "csvqa_trace": "csvqa_trace.json","generated_model": "model.md", "solve_code": "solve.py", "csvqa_observation": "data_overview.md"}


def _sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def materialize_record(csv_path, record):
    """Load a saved record's artifacts so save_record can copy them to another result folder."""
    root, row = Path(csv_path).parent, dict(record)
    for field in RECORD_ARTIFACT_FILES:
        relative = row.pop(field + "_path", None)
        expected_hash = row.pop(field + "_sha256", None)
        if not relative and not expected_hash and row.get("record_status") == "error":
            continue  # A failed stage may never have produced this artifact.
        artifact = root / relative if relative else None
        if artifact is None or not artifact.is_file() or not expected_hash:
            return None
        try:
            raw = artifact.read_bytes()
            if _sha256_bytes(raw) != expected_hash:
                return None
            row[field] = raw.decode("utf-8")
        except (OSError, UnicodeError):
            return None
    relative = row.pop("solution_path", None)
    expected_hash = row.pop("solution_sha256", None)
    artifact = root / relative if relative else None
    if artifact is None or not artifact.is_file() or not expected_hash:
        return None
    try:
        raw = artifact.read_bytes()
        if _sha256_bytes(raw) != expected_hash:
            return None
        solution = json.loads(raw.decode("utf-8"))
    except (OSError, UnicodeError, json.JSONDecodeError):
        return None
    if not isinstance(solution, dict) or "variables" not in solution:
        return None
    row["final_solution"] = solution["variables"]
    return row


def reusable_record(csv_path, record, fingerprint):
    """Resume intact recorded attempts, including failures, without selective retries."""
    if (not record or record.get("record_status") not in {"completed", "error"}
            or record.get("cache_fingerprint") != fingerprint):
        return None
    return materialize_record(csv_path, record)


def save_record(csv_path, record):
    """Save readable case files, then atomically update the small results CSV."""
    path = Path(csv_path)
    # Percent-encoding preserves distinct IDs without treating them as paths.
    identity = json.dumps(record_key(record), ensure_ascii=False, separators=(",", ":"), default=str)
    artifact_id = hashlib.sha256(identity.encode("utf-8")).hexdigest()[:16]
    folder = (path.parent / (path.stem + "_cases") / (record.get("forced_route") or "AUTO")
              / quote(str(record["problem_id"]), safe="").replace(".", "%2E") / artifact_id)
    folder.mkdir(parents=True, exist_ok=True)
    row = dict(record)
    with _RESULT_LOCK, path.with_suffix(".csv.lock").open("a+") as lock:
        fcntl.flock(lock.fileno(), fcntl.LOCK_EX)
        for field, filename in RECORD_ARTIFACT_FILES.items():
            content = row.pop(field, None)
            if content is not None:
                content = str(content)
                _write_text_atomic(folder / filename, content, "utf-8")
                row[field + "_path"] = str((folder / filename).relative_to(path.parent))
                row[field + "_sha256"] = _sha256_bytes(content.encode("utf-8"))
        if "final_solution" in row:
            solution = {"status": "OPTIMAL" if row.get("final_ok") else "ERROR",
                        "objective": row.get("final_objective"), "variables": row.pop("final_solution"),
                        "error": row.get("execution_error")}
            solution_text = json.dumps(solution, ensure_ascii=False, indent=2, allow_nan=False)
            _write_text_atomic(folder / "solution.json", solution_text, "utf-8")
            row["solution_path"] = str((folder / "solution.json").relative_to(path.parent))
            row["solution_sha256"] = _sha256_bytes(solution_text.encode("utf-8"))
        records = [old for old in load_records(path) if record_key(old) != record_key(row)] + [row]
        _write_text_atomic(path, pd.DataFrame(records).to_csv(index=False), "utf-8-sig")
    return records


def _source_fingerprint():
    if not NOTEBOOK_PATH.is_file():
        raise FileNotFoundError(
            f"Configured notebook for cache fingerprint does not exist: {NOTEBOOK_PATH}. "
            "Set LEAN_LLM_OPT_NOTEBOOK to the notebook being executed."
        )
    notebook = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
    if notebook.get("metadata", {}).get("experiment_provenance", {}).get("method") != EXPERIMENT_METHOD:
        raise ValueError(
            "LEAN_LLM_OPT_NOTEBOOK must point to the Few-shot Only notebook being executed, "
            "not the full baseline or another method."
        )
    source = "\n".join("".join(cell["source"]) for cell in notebook["cells"]
                       if cell["cell_type"] == "code" and "experiment" not in cell.get("metadata", {}).get("tags", []))
    digest = hashlib.sha256((CACHE_SCHEMA_VERSION + "\n" + str(NOTEBOOK_PATH) + "\n" + source).encode())
    for path in sorted((PROJECT_ROOT / "Large_Scale_Or_Files").rglob("*.csv")):
        digest.update(str(path.relative_to(PROJECT_ROOT)).encode())
        digest.update(path.read_bytes())
    worker_path = PROJECT_ROOT / "solver_worker.py"
    if worker_path.is_file():
        digest.update(b"solver_worker.py\0")
        digest.update(worker_path.read_bytes())
    return digest.hexdigest()


def case_fingerprint(case, route=None, *, source_fingerprint=None):
    fields = {"cache_schema": CACHE_SCHEMA_VERSION,
              "source": source_fingerprint if source_fingerprint is not None else _source_fingerprint(),
              "model": MODEL_SNAPSHOT,
              "embedding_model": EMBEDDING_MODEL, "data_access": CSVQA_MODE,
              "experiment_method": EXPERIMENT_METHOD,
              "nrm_retry_on_truncation": NRM_RETRY_ON_TRUNCATION,
              "query": str(case["Query"]), "forced_route": route}
    # Frozen labels are inputs to automatic routing; changed labels must invalidate reuse.
    # Forced-route fingerprints do not depend on an unused classifier prediction.
    if route is None:
        fields["classification"] = (
            {"source": "baseline_results_csv",
             "predicted_label": normalize_problem_class(case["cached_predicted_label"])}
            if "cached_predicted_label" in case else {"source": "online_classifier"}
        )
    fields.update({
        "solver_policy": SOLVER_POLICY,
        "execution_limits": EXECUTION_LIMITS,
    })
    digest = hashlib.sha256(json.dumps(fields, sort_keys=True).encode())
    for raw in normalize_data_address(case.get("dataset_address")).splitlines():
        path = Path(raw)
        digest.update(str(path).encode())
        digest.update(path.read_bytes() if path.is_file() else b"MISSING_FILE")
    return digest.hexdigest()


def error_record(case, exc, forced_route=None):
    return {
        **case_fields(case), "assigned_route": forced_route, "predicted_label": None,
        **getattr(exc, "pipeline_context", {}),
        "experiment_mode": "forced" if forced_route else "automatic", "forced_route": forced_route,
        "final_ok": False, "final_objective": None, "final_solution": None,
        "execution_error": f"{type(exc).__name__}: {exc}", "execution_error_type": type(exc).__name__,
        "record_status": "error", "cache_source": "computed",
    }


## 17. Shared test runner

`run_test(..., classification_csv=...)` reuses automatic baseline predictions after
ID, question, and automatic/forced provenance checks. `classification_csv=None`
uses the original classifier. Missing or invalid cached predictions fail before
model calls; there is no silent online-classifier fallback for a partial cache.

Only Few-shot Only outputs may be resumed or copied for oracle AUTO reuse. The
classification source CSV is not a source of models, code, or solutions and cannot
be used as the output path. For a new experiment, use a fresh version directory.
Forced routes do not call the classifier; automatic oracle reuse remains subject
to the full Few-shot Only cache fingerprint and intact-artifact checks.


In [18]:
def reset_run_caches():
    """Rebuild RAG state between runs, while retaining reuse within each run."""
    _get_classification_agent.cache_clear()
    _get_classifier_retriever.cache_clear()
    _get_rag_store.cache_clear()
    _load_rag_table.cache_clear()


def run_test(test, *, output_csv=RESULTS_DIR / "results.csv", routes=None, reuse_auto_csv=None,
             reuse_completed=True, continue_on_error=True, rel_tol=1e-4, abs_tol=1e-4,
             classification_csv=None):
    """Resume the same immutable round; use a fresh output directory for a new round."""
    frame = load_benchmark(test) if isinstance(test, (str, Path)) else prepare_cases(test)
    if classification_csv is not None:
        classification_path = Path(classification_csv).expanduser()
        classification_path = (classification_path if classification_path.is_absolute()
                               else PROJECT_ROOT / classification_path).resolve()
        if Path(output_csv).expanduser().resolve() == classification_path:
            raise ValueError("Output CSV must not overwrite the baseline classification CSV")
        if routes is None or reuse_auto_csv is not None:
            frame = attach_cached_classifications(frame, classification_path)
    selected_routes = [None] if routes is None else [normalize_route(route) for route in routes]
    if not selected_routes or len(selected_routes) != len(set(selected_routes)):
        raise ValueError("Provide a non-empty list of distinct routes")
    if routes is not None and frame["dataset_address"].eq("").any():
        raise ValueError("Forced-route tests require case CSV inputs; use routes=None for query-only tests")
    existing = load_records(output_csv)
    if existing and not reuse_completed:
        raise ValueError("Use a fresh round directory; existing attempts cannot be overwritten")
    cached = {record_key(row): row for row in existing}
    auto_cached = ({record_key(row): row for row in load_records(reuse_auto_csv)}
                   if reuse_completed and reuse_auto_csv else {})
    # Fingerprint every run before saving records so a first run is reusable later.
    reset_run_caches()
    source = _source_fingerprint()
    results, api_checked = [], False
    total = len(frame) * len(selected_routes)
    for case in frame.to_dict("records"):
        for route in selected_routes:
            key = (case["problem_id"], str(case["Query"]), case["dataset_address"], route or "AUTO")
            fingerprint = case_fingerprint(case, route, source_fingerprint=source)
            previous = cached.get(key)
            if previous and previous.get("cache_fingerprint") != fingerprint:
                raise ValueError("Code/data changed; use a fresh version directory")
            record = reusable_record(output_csv, previous, fingerprint)
            if previous and record is None:
                raise ValueError("Recorded artifacts are missing/corrupt; do not replace the attempt")
            if record is not None:
                record["cache_source"] = "csv"
            elif route is not None and route == case.get("true_route"):
                automatic = auto_cached.get(key[:-1] + ("AUTO",))
                if automatic and automatic.get("assigned_route") == route:
                    auto_fingerprint = case_fingerprint(case, source_fingerprint=source)
                    record = reusable_record(reuse_auto_csv, automatic, auto_fingerprint)
                    if record is not None:
                        record["cache_source"] = "oracle_auto_reuse"
            if record is not None:
                record.update(**case_fields(case), forced_route=route,
                              experiment_mode="forced" if route else "automatic")
            else:
                # A missing credential is a run configuration error, not 101 case failures.
                if not api_checked:
                    require_api_key()
                    api_checked = True
                try:
                    record = execute_pipeline_case(case, forced_route=route)
                except Exception as exc:
                    record = error_record(case, exc, route)
                    print(f"{case['problem_id']} / {route or 'AUTO'} "
                          f"failed at {record.get('pipeline_stage', 'setup')}: {record['execution_error']}")
                    if not continue_on_error:
                        record = finalize_record(record, rel_tol=rel_tol, abs_tol=abs_tol)
                        record.update(solver_policy=SOLVER_POLICY,
                                      execution_limits=json.dumps(EXECUTION_LIMITS, sort_keys=True))
                        record["cache_fingerprint"] = fingerprint
                        save_record(output_csv, record)
                        raise
            record = finalize_record(record, rel_tol=rel_tol, abs_tol=abs_tol)
            record.update(solver_policy=SOLVER_POLICY,
                          execution_limits=json.dumps(EXECUTION_LIMITS, sort_keys=True))
            record["cache_fingerprint"] = fingerprint
            save_record(output_csv, record)
            results.append(record)
            print(f"[{len(results)}/{total}] {case['problem_id']} / {route or 'AUTO'}: "
                  f"{record['record_status']} [{record.get('cache_source')}]")
    return results


def evaluation_summary(records):
    frame = pd.DataFrame(records)
    if frame.empty:
        return pd.DataFrame(columns=["metric", "correct", "total", "accuracy"])
    rows = []
    if frame["experiment_mode"].eq("forced").all():
        oracle = frame.loc[
            frame["forced_route"].eq(frame["true_route"])
            & frame["solution_correct"].notna()
        ].drop_duplicates("problem_id")
        for label, correct, total in (
                ("Oracle routing accuracy", int(oracle["solution_correct"].eq(True).sum()), oracle["problem_id"].nunique()),
                ("Solved accuracy", int(frame["final_ok"].eq(True).sum()), len(frame))):
            rows.append({"metric": label, "correct": correct, "total": total,
                         "accuracy": correct / total if total else None})
        return pd.DataFrame(rows)
    for field, label in (("final_ok", "Solved"), ("classification_correct", "Classification"), ("solution_correct", "Objective match")):
        values = frame[field].dropna() if field in frame else pd.Series(dtype=bool)
        total, correct = len(values), int(values.eq(True).sum())
        rows.append({"metric": label, "correct": correct, "total": total,
                     "accuracy": correct / total if total else None})
    return pd.DataFrame(rows)


def classification_confusion(records):
    frame = pd.DataFrame(records)
    frame = frame.loc[frame["true_label"].notna() & frame["experiment_mode"].eq("automatic")]
    predicted = frame["predicted_label"].fillna("Unclassified")
    columns = CLASS_LABELS + (["Unclassified"] if predicted.eq("Unclassified").any() else [])
    return pd.crosstab(frame["true_label"], predicted).reindex(index=CLASS_LABELS, columns=columns, fill_value=0)


def accuracy_by_class(records):
    frame = pd.DataFrame(records)
    grouped = frame.groupby("true_label").agg(
        total=("problem_id", "size"), classification_correct=("classification_correct", "sum"),
        classification_scored=("classification_correct", "count"), solution_correct=("solution_correct", "sum"),
        solution_scored=("solution_correct", "count"))
    for metric in ("classification", "solution"):
        grouped[metric + "_accuracy"] = grouped[metric + "_correct"] / grouped[metric + "_scored"].replace(0, np.nan)
    return grouped.reindex(CLASS_LABELS).rename_axis("true_label")


def forced_route_matrix(records):
    frame = pd.DataFrame(records)
    grouped = frame.groupby(["true_route", "assigned_route"])["solution_correct"].agg(correct="sum", total="count")
    formatted = grouped.apply(lambda row: f"{int(row.correct)}/{int(row.total)} ({row.correct / row.total:.2%})" if row.total else "Not scored", axis=1)
    return formatted.unstack("assigned_route").reindex(index=WORKFLOW_ROUTES, columns=WORKFLOW_ROUTES)


def accuracy_by_route(records):
    grouped = pd.DataFrame(records).groupby("assigned_route")["solution_correct"].agg(correct="sum", total="count")
    grouped["accuracy"] = grouped["correct"] / grouped["total"].replace(0, np.nan)
    return grouped.reindex(WORKFLOW_ROUTES)


def report_results(records, output_dir):
    """Display and save the tables appropriate to the experiment."""
    tables = {"summary": evaluation_summary(records)}
    if records:
        if all(row.get("experiment_mode") == "automatic" for row in records):
            tables.update(classification_confusion=classification_confusion(records), accuracy_by_class=accuracy_by_class(records))
        else:
            tables.update(route_matrix=forced_route_matrix(records), accuracy_by_route=accuracy_by_route(records))
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    for name, table in tables.items():
        print(name.replace("_", " ").title())
        shown = table.copy()
        for column in shown.columns:
            if str(column).endswith("accuracy"):
                shown[column] = shown[column].map(lambda value: f"{value:.2%}" if pd.notna(value) else "Not scored")
        display(shown)
        table.to_csv(output_dir / f"{name}.csv", index=table.index.name is not None, encoding="utf-8-sig")
    return tables


## 18. Test Large-scale-or: automatic (101 cases)

Run Few-shot Only with the full baseline's automatic predicted labels. Baseline gold
labels/models/solver outputs are not loaded through the classification reader.
Only prediction identity/provenance is reused; modeling and solving run anew.
Classification accuracy is inherited from the frozen predictions, not a newly
measured classifier. Modeling/solution metrics evaluate this ablation.


In [19]:
RUN_AUTOMATIC = True
if RUN_AUTOMATIC:
    automatic_results = run_test(
        BENCHMARK_PATH,
        output_csv=RESULTS_DIR / "automatic/results.csv",
        classification_csv=CLASSIFICATION_RESULTS_PATH,
    )
    report_results(automatic_results, RESULTS_DIR / "automatic")


/var/folders/xt/y7vvngb13nnc6smm6498f3580000gn/T/ipykernel_18877/3748439118.py:16: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the [LangGraph documentation](https://langchain-ai.github.io/langgraph/) as well as guides for [Migrating from AgentExecutor](https://python.langchain.com/docs/how_to/migrate_agent/) and LangGraph's [Pre-built ReAct agent](https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/).
  agent = initialize_agent(




> Entering new AgentExecutor chain...
Question: Amazon operates multiple distribution centers that are responsible for supplying essential goods daily to various customer groups. The daily demand for each customer group is provided in "customer_demand.csv", while the daily supply capacity of the distribution centers is detailed in "supply_capacity.csv". The transportation cost per unit of goods from each distribution center to each customer group is recorded in "transportation_costs.csv". The objective is to determine the quantity of goods to be transported from each distribution center to each customer group, ensuring that all customer demands are met without exceeding the supply capacity of any distribution center, while minimizing the total transportation cost.

Thought: I need to retrieve all data from the three CSV files: customer_demand.csv, supply_capacity.csv, and transportation_costs.csv, preserving all identifiers and coefficients in source order.

Action: CSVQA

Action Inp

/var/folders/xt/y7vvngb13nnc6smm6498f3580000gn/T/ipykernel_18877/2785705467.py:107: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  abstract_model_chain = LLMChain(llm=llm2, prompt=abstract_prompt)
/var/folders/xt/y7vvngb13nnc6smm6498f3580000gn/T/ipykernel_18877/2785705467.py:109: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  abstract_model_plan = abstract_model_chain.run(


[Observation]:
[Abstract Model Plan START]
1.  **Analyze Query:** The user wants to determine the minimum number of drivers and crew members to assign to a bus route, such that for each hour of the day, the number of staff on duty (each working a continuous 4-hour shift starting at the beginning of an hour) meets or exceeds the required number for that hour, as specified in the CSV.
2.  **Identify Model Type:** Based on the query, this is a Linear Programming (LP) covering problem (specifically, a cyclic set covering problem due to the 24-hour wrap-around).
3.  **Define Index Sets:** The primary index is the set of time periods (hours), indexed by \( t \) for \( t = 1, 2, ..., 24 \), corresponding to the 24 hourly shifts in the CSV.
4.  **Define Decision Variables:**
    -   `x[t]` = Number of drivers and crew members starting a 4-hour shift at hour \( t \). Type: GRB.INTEGER (nonnegative integer).
5.  **Identify Parameters (from Schema):**
    -   Required staff per hour: from column 

,metric,correct,total,accuracy
0,Solved,81,101,80.20%
1,Classification,94,101,93.07%
2,Objective match,78,101,77.23%


Classification Confusion


predicted_label,TP,NRM,RA,FLP,AP,Mixture,Others
true_label,,,,,,,
TP,9,0,0,0,0,0,0
NRM,0,25,0,0,0,0,0
RA,0,0,22,0,0,0,0
FLP,0,0,0,14,0,0,0
AP,0,0,0,0,5,0,0
Mixture,0,0,4,0,0,12,2
Others,0,0,1,0,0,0,7


Accuracy By Class


,total,classification_correct,classification_scored,solution_correct,solution_scored,classification_accuracy,solution_accuracy
true_label,,,,,,,
TP,9,9,9,8,9,100.00%,88.89%
NRM,25,25,25,10,25,100.00%,40.00%
RA,22,22,22,21,22,100.00%,95.45%
FLP,14,14,14,13,14,100.00%,92.86%
AP,5,5,5,5,5,100.00%,100.00%
Mixture,18,12,18,14,18,66.67%,77.78%
Others,8,7,8,7,8,87.50%,87.50%


## 19. Optional: all routes (606 cases)

Disabled by default. Every problem is sent through all six routes without a classifier
call. Only matching AUTO artifacts from this Few-shot Only experiment may be reused
for an oracle route. The baseline CSV supplies labels only; it is never passed as
`reuse_auto_csv`. Classification attachment supports the matching AUTO fingerprint.


In [20]:
RUN_FORCED_ROUTES = False
if RUN_FORCED_ROUTES:
    forced_route_results = run_test(
        BENCHMARK_PATH, output_csv=RESULTS_DIR / "forced/results.csv",
        routes=WORKFLOW_ROUTES,
        reuse_auto_csv=RESULTS_DIR / "automatic/results.csv",
        classification_csv=CLASSIFICATION_RESULTS_PATH,
    )
    report_results(forced_route_results, RESULTS_DIR / "forced")


## 20. Optional: IndustryOR or another dataset

Disabled by default. Use `classification_csv=None` to run the original classifier,
or pass the matching automatic baseline results for this dataset. Do not pass the
101-case classification file to an unrelated benchmark.


In [21]:
RUN_OTHER_DATASET = False
if RUN_OTHER_DATASET:
    other_results = run_test(
        OTHER_BENCHMARK_PATH, output_csv=RESULTS_DIR / "other/results.csv",
        classification_csv=None,
    )
    report_results(other_results, RESULTS_DIR / "other")


## 21. Optional single-case check

Disabled by default. To test one case first, set RUN_AUTOMATIC=False above, then
enable this cell. It uses the same frozen 101-case predictions and stops at the
first error. Use a fresh single-case output directory for a new attempt.


In [22]:
RUN_SINGLE_CASE = False
if RUN_SINGLE_CASE:
    single_case_results = run_test(
        load_benchmark().head(1), output_csv=RESULTS_DIR / "single_case/results.csv",
        reuse_completed=False, continue_on_error=False,
        classification_csv=CLASSIFICATION_RESULTS_PATH,
    )
    report_results(single_case_results, RESULTS_DIR / "single_case")


## Additional benchmark loaders

Retain the original data and stable Variant/sheet IDs. These optional datasets
are disabled by default and use online classification unless their own matching
`classification_csv` is explicitly supplied. No gold model enters generation.


In [23]:
# Additional benchmark loader module: relocated data, variants, and redundant-column sheets.

# Keep the original implementation across repeated notebook execution.
_BASE_PREPARE_CASES = globals().get("_BASE_PREPARE_CASES", prepare_cases)

if not getattr(prepare_cases, "_relocated_dataset_wrapper", False):
    def prepare_cases(test, *, dataset_root=None):
        if dataset_root is None:
            return _BASE_PREPARE_CASES(test)
        frame = test.copy()
        address_column = "dataset_address" if "dataset_address" in frame else "Dataset_address"
        def resolve_relocated(value):
            paths = []
            root = Path(dataset_root).resolve()
            for raw in normalize_data_address(value).splitlines():
                path = Path(raw)
                standard = path if path.is_absolute() else PROJECT_ROOT / path
                if standard.is_file():
                    paths.append(str(standard.resolve())); continue
                candidates = [root/path, root.joinpath(*path.parts[1:])]
                matches = list(dict.fromkeys(p.resolve() for p in candidates if p.is_file()))
                if len(matches) != 1:
                    raise ValueError(f"Cannot uniquely resolve relocated data: {raw}")
                paths.append(str(matches[0]))
            return "\n".join(paths)
        frame[address_column] = frame[address_column].map(resolve_relocated)
        return _BASE_PREPARE_CASES(frame)
    prepare_cases._relocated_dataset_wrapper = True

# To run only this benchmark, execute the preceding definition cells, then this cell.
RUN_VARIANTS = False
VARIANTS_FILE = PROJECT_ROOT / "benchmark_dataset/questions.csv"
VARIANTS_OUTPUT_DIR = RESULTS_DIR / "additional_benchmarks/variants"


def load_variants_for_baseline(path):
    frame = load_benchmark(path)
    problem_ids = []
    for address in frame["dataset_address"]:
        paths = [Path(line) for line in normalize_data_address(address).splitlines()]
        if not paths or any(not source.is_file() for source in paths):
            raise FileNotFoundError(f"Missing variants data files: {address}")
        ids = {part for source in paths for part in source.parts
               if re.fullmatch(r"Variant[0-9]+", part)}
        if len(ids) != 1:
            raise ValueError(f"Cannot determine a unique Variant ID: {address}")
        problem_ids.append(next(iter(ids)))
    if len(problem_ids) != len(set(problem_ids)):
        raise ValueError("Duplicate Variant IDs; check questions.csv")
    frame["problem_id"] = problem_ids
    return frame

RUN_REDUNDANT_COLUMNS = False
COLUMNS_FILE = PROJECT_ROOT / "redundancy_complete/redundant_instances.xlsx"
COLUMN_SHEETS = [f"{pct}-S{seed}" for pct in ["50pct", "100pct", "200pct"] for seed in [1, 2, 3]]  # Select one or more sheets.
# Last three sheets: COLUMN_SHEETS = ["200pct-S1", "200pct-S2", "200pct-S3"]
# All nine sheets: COLUMN_SHEETS = [f"{pct}-S{seed}" for pct in ["50pct", "100pct", "200pct"] for seed in [1, 2, 3]]
COLUMNS_OUTPUT_DIR = RESULTS_DIR / "additional_benchmarks/columns"


def load_redundant_sheets_for_baseline(path, sheet_names):
    path = Path(path).expanduser()
    path = path if path.is_absolute() else PROJECT_ROOT / path
    selected = [sheet_names] if isinstance(sheet_names, str) else list(sheet_names)
    if not selected or any(not isinstance(name, str) or not name.strip() for name in selected):
        raise ValueError("COLUMN_SHEETS must contain at least one non-empty sheet name")
    if len(selected) != len(set(selected)):
        raise ValueError("COLUMN_SHEETS must not contain duplicate sheet names")
    frames = {}
    with pd.ExcelFile(path) as workbook:
        missing = [name for name in selected if name not in workbook.sheet_names]
        if missing:
            raise ValueError(f"Unknown sheets: {missing}; available sheets: {workbook.sheet_names}")
        for name in selected:
            # dataset_root resolves legacy directory names under the current workbook directory.
            frame = prepare_cases(pd.read_excel(workbook, sheet_name=name), dataset_root=path.parent)
            frame["benchmark_sheet"] = name
            frame["problem_id"] = name + "/" + frame["problem_id"]
            for address in frame["dataset_address"]:
                sources = [Path(line) for line in normalize_data_address(address).splitlines()]
                if not sources or any(not source.is_file() for source in sources):
                    raise FileNotFoundError(f"Missing data files for {name}: {address}")
            frames[name] = frame
    return frames


In [24]:
# Additional benchmark runner module.

RUN_VARIANTS = False
if RUN_VARIANTS:
    rows = load_variants_for_baseline(VARIANTS_FILE)
    results = run_test(rows, output_csv=VARIANTS_OUTPUT_DIR / "results.csv", classification_csv=None)
    report_results(results, RESULTS_DIR / "additional_benchmarks/variants")

RUN_REDUNDANT_COLUMNS = False
if RUN_REDUNDANT_COLUMNS:
    by_sheet = load_redundant_sheets_for_baseline(COLUMNS_FILE, COLUMN_SHEETS)
    for sheet_name, rows in by_sheet.items():
        folder = RESULTS_DIR / "additional_benchmarks/columns" / sheet_name
        results = run_test(rows, output_csv=folder / "results.csv", classification_csv=None)
        report_results(results, folder)
